# Inner-speech EEG channel reduction: two evaluation boundaries

Compare **128, 64, 32 and 16 available channels** using fixed spatially distributed, nested electrode sets. Subjects are evaluated independently, pooling their available INNER SPEECH sessions. Trials, target labels and shuffled stratified 10-fold plans are identical across channel counts and boundaries within each experiment.

- **PAPER_ORDER — feature selection before CV; potential leakage:** full-subject labels select features before classifier CV. This branch deliberately contains held-out-label leakage. It is a diagnostic ordering experiment, not a valid deployment estimate or a claim about published author code.
- **NESTED_SAFE:** selection, scaling and any model tuning use training rows only. Exactly one outer OOF prediction is collected per original trial. Fixed-K/fixed-parameter runs have no inner search; optional SVM tuning uses inner CV.

Initial configuration: Subject 01, all four channel counts, MRMR/SVM, K=590, both boundaries, VMD off. Five initial shuffled-label controls run at 128 and 16 channels. Change `subjects=None` for the full available cohort. More classifiers and ANOVA are configurable. This stage implements ML; the saved fixed electrode sets can support later DL/pretrained comparisons.

This is a new standalone notebook: no old saved folds, feature exports or local project files are required. Source EEG and existing notebooks are not changed.


## 1. Dependencies and source provenance

Required installed packages: NumPy, pandas, SciPy, scikit-learn, matplotlib, MNE and PyWavelets. No internet/pip command is run. LightGBM is optional and explicitly skipped if absent. The existing audited loader, current feature extractor, MRMR quotient and evaluation engine are embedded below.

The processed 128-channel FIF data already received provider preprocessing. Selecting fewer channels here masks those processed recordings; it does not reproduce a newly acquired16-electrode recording or recalculate low-density reference/ICA preprocessing.

Reference context: Tasin et al., [DOI 10.1109/ACCESS.2025.3644494](https://doi.org/10.1109/ACCESS.2025.3644494). The exact 191-feature supplementary definitions remain unavailable. Current combined features give120/channel; the optional documented paper-inspired approximation gives 152/channel and is not author-exact.

Evaluation references: [feature-selection leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage-during-pre-processing), [nested tuning](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).


In [ ]:
"""Subject-specific paper-order diagnostic and training-only evaluation."""
from pathlib import Path
from datetime import datetime, timezone
from fractions import Fraction
from types import SimpleNamespace
from time import perf_counter
from zipfile import ZipFile, ZIP_DEFLATED
from itertools import combinations
from importlib.metadata import version, PackageNotFoundError
import hashlib
import importlib.util
import json
import platform
import re
import warnings
import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.signal import welch
from scipy.integrate import trapezoid
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, recall_score
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
import pywt
BANDS = ((1.,4.), (4.,8.), (8.,13.), (13.,30.), (30.,45.))
PAPER_LABEL = 'PAPER_ORDER — feature selection before CV; potential leakage'
SAFE_LABEL = 'NESTED_SAFE'
LABELS = [0, 1, 2, 3]
REFERENCES = {
    ('mrmr', 10, 'svm'): (.6653, .6709),
    ('anova', 10, 'lgbm'): (.4594, .4630),
    ('mrmr', 590, 'svm'): (.7631, None),
    ('mrmr', 590, 'lda'): (.7191, None),
    ('mrmr', 590, 'etc'): (.7593, None),
    ('mrmr', 590, 'rf'): (.7342, None),
    ('mrmr', 590, 'lr'): (.7789, None),
    ('mrmr', 590, 'lgbm'): (.7533, None),
    ('mrmr', 590, 'lr5'): (.8113, .8112),
}
_sklearn_mutual_info_classif = mutual_info_classif
"""Fixed spatial channel sets and paired evaluation-boundary comparisons."""
from copy import deepcopy
CHANNEL_EXTENSION_VERSION='channel-reduction-dual-1.0'
CHANNEL_MODEL_NAMES={'svm','lda','etc','rf','lr','lgbm','lr5'}
CHANNEL_BOUNDARY_COLORS={'PAPER_ORDER':'#d95f02','NESTED_SAFE':'#1b9e77'}
ENGINE_ID='3fcd049c11cab497276a365b5d7a5116fed7ea0d19104739446ae4fb4226c083'
SOURCE_INSPECTION={'current_source': 'Thinking_Out_Loud_Within_Session_ML_Ensemble / audited current source', 'current_source_sha256': 'c168a2f9351deb2b19d6060d219a4233aacdfb688a933d83ecf4a5f941994511', 'current_wavelet_sha256': 'e7d884d0c48ea58ba5102a0bfe27497b84a8eae01e4c7b064376d1bbdc814b82', 'paper_inspired_source': 'C:\\Users\\User\\Programming\\Learning\\python\\braincodecamp\\braincodecamp2026\\paper-reproduce.ipynb', 'paper_inspired_source_sha256': '0622a815f14cbd30f173a45f07a4859174a2eba9807eddbc827d3adb96add733', 'paper_exact_191_available': False, 'paper_pdf': 'C:\\Users\\User\\Programming\\Learning\\python\\braincodecamp\\braincodecamp2026\\presentations\\papers\\Ensemble_Machine_Learning_Model_for_Inner_Speech_Recognition_A_Subject-Specific_Investigation.pdf', 'doi': '10.1109/ACCESS.2025.3644494', 'current_default_reason': 'The saved compare-wavelet-ml-mrmr-anova run uses combined features.', 'loader_adaptation': 'Validated derivatives loader; retain pooled EEG and session IDs; crop by FIF times.', 'no_existing_files_modified': True, 'channel_configuration': 'Canonical BioSemi128 farthest-point spatial rule, matching inspected channel_subsets; nested sets frozen without EEG/labels.', 'previous_channel_notebook': 'No prior notebook is imported at runtime; code is embedded.', 'previous_audited_engine_id': 'f1c2025bfe88cbb552c1ec9e342a8d14d0ce5970f26e5d035e930ca65902f27b'}
from IPython.display import display, FileLink
print("Python:",platform.python_version())
print("Engine:",ENGINE_ID[:16])


## 2. Edit this configuration cell

The most useful controls are `subjects`, `channel_counts`, `classifiers`, `selectors`, `k_features` and `run_permutation_controls`.

- `subjects=[1]` checks the implementation on one subject; `None` discovers all subjects.
- Keep128 in `channel_counts` for the paired reference. Default spatial geometry sets are identical across subjects/models/modes. `channel_method="manual"` requires explicit nested name lists for every requested count, including128.
- Start `classifiers=["svm"]`; later use `["svm","lda","lr","lr5"]` or add `etc`,`rf`,`lgbm`. `models` holds their hyperparameters; `classifiers` selects which models run.
- `selectors=["mrmr"]` initially; add `"anova"` for a matched selector comparison.
- Keep K=590 fixed initially. It fits within the current 16-channel pool of 1,920 features. If you choose bandpower-only features, the 16-channel pool has only 80 features, so you must lower K; the notebook raises an error instead of silently clipping it.
- Five permutations are an initial diagnostic, not a formal significance test. With all 10 subjects selected, controls also run on all 10 when `permutation_subjects=None`; use `[1]` to keep the first control run small.
- VMD is frozen off in this controlled comparison. Changing preprocessing requires a separate protocol.

Settings match the previous audited ML comparison, including RBF SVM C=1/gamma scale and F-stat/Pearson MRMR. These are implementation choices, not verified paper hyperparameters. Avoid choosing settings from outer OOF plots.


In [ ]:
CONFIG = {
    "data_root": None,  # Discover one derivatives dataset under /kaggle/input; or set an explicit Path.
    "output_root": Path("/kaggle/working/inner_speech_channel_reduction"),
    "subjects": [1],    # None = all discovered subjects; each gets its own models.
    "sessions": None,   # Pool all available INNER SPEECH sessions within each subject.
    "channel_counts": [128, 64, 32, 16],
    "channel_method": "spatial",       # spatial / manual
    "manual_channel_sets": {},         # Example format: {128: [...],64: [...],32: [...],16: [...]}
    "boundaries": ["PAPER_ORDER", "NESTED_SAFE"],
    "selectors": ["mrmr"],            # Add "anova" if wanted.
    "classifiers": ["svm"],           # svm, lda, etc, rf, lr, lgbm, lr5
    "k_features": 590,
    "run_main": True,
    "run_permutation_controls": True,
    "n_permutations": 5,
    "permutation_subjects": None,      # All selected subjects; use [1] for a small cohort control.
    "permutation_channel_counts": [128, 16],
    "permutation_selectors": ["mrmr"],
    "permutation_classifiers": ["svm"],
    "seed": 42,
    "outer_folds": 10,
    "inner_folds": 3,
    "action_window_mne": (0.5, 3.0),   # Audited FIF epoch clock; exclusive endpoint.
    "feature_mode": "current",        # current / paper_inspired
    "current_features": {
        "feature_set": "combined",    # combined / eight_window / bandpower
        "n_windows": 8, "wavelet": "db2", "wavelet_level": 3,
        "ar_order": 4, "feature_sfreq": 128., "wavelet_bandpass": [2.,45.],
    },
    "paper_inspired_features": {"wavelet": "db4", "wavelet_level": 5, "spectral_high": 100.},
    "vmd_mode": "off",                # Frozen off; other modes require a separate study.
    "vmd_manual_channels": {}, "vmd_robust_z": 4.,
    "vmd_parameters": {"alpha": 2000., "tau": 0., "dc": 0, "init": 1, "tol": 1e-7},
    "mrmr_relevance": "f_stat",        # f_stat / mi; MI is slower and estimator is an assumption.
    "mrmr_corr_floor": 1e-6,
    "tune_svm": False,                 # Optional training-only innerC search; disable for initial controls.
    "svm_inner_grid": {"C": [.1,1.,10.], "kernel": ["rbf"], "gamma": ["scale"]},
    "models": {
        "svm": {"C": 1., "kernel": "rbf", "gamma": "scale", "cache_size": 512},
        "lda": {"solver": "lsqr", "shrinkage": "auto"},
        "etc": {"n_estimators": 300, "max_features": "sqrt"},
        "rf": {"n_estimators": 300, "max_features": "sqrt"},
        "lr": {"C": 1., "solver": "lbfgs", "max_iter": 5000},
        "lgbm": {"n_estimators": 300, "learning_rate": .05, "num_leaves": 15,
                 "deterministic": True, "force_col_wise": True},
    },
    "lr5_members": [  # Paper-inspired fiveLR configs; exact author settings are not reported.
        {"C": .1, "penalty": "l2", "solver": "lbfgs"},
        {"C": 1., "penalty": "l2", "solver": "lbfgs"},
        {"C": 10., "penalty": "l2", "solver": "lbfgs"},
        {"C": 1., "penalty": "l1", "solver": "liblinear"},
        {"C": 1., "penalty": "elasticnet", "solver": "saga", "l1_ratio": .5},
    ],
    "lr5_max_iter": 5000,
    "n_jobs": 1, "blas_threads": 1,
    "resume": True,
    "make_plots": True,
    "plot_individual_subjects": True,
    "show_channel_names": False,
    "class_names": {0: "Up",1: "Down",2: "Right",3: "Left"},
}


## 3. Reused trial-local feature extraction

The current extractor is embedded unchanged: eight windows, Burg AR4, db2 level3 DWT entropy/log variance and filtered-power fusion. Every original trial remains one row. The optional existing 152-feature approximation includes signal/envelope statistics, Welch PSD descriptors and db4 level5 DWT summaries; its geometric-length descriptor is not asserted to equal the unavailable paper fractal formula.

Features are computed once from the processed 128-channel recordings, without labels. Both provided extractors calculate features independently per channel, so masking columns by the feature-to-channel schema is equivalent to retaining those channels' descriptors. Supervised selection is always rerun after this mask.


In [ ]:
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str,
                                    separators=(',', ':')).encode()).hexdigest()

def array_digest(value):
    value = np.ascontiguousarray(value)
    h = hashlib.sha256(str((value.shape, value.dtype.str)).encode())
    h.update(memoryview(value).cast('B'))
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def bandpower_features(X, sfreq):
    n = min(int(round(sfreq)), X.shape[-1])
    if sfreq / 2 < 45 or n < 16:
        raise ValueError('Insufficient sampling rate or epoch length for declared bands.')
    freqs, density = welch(X, fs=sfreq, window='hann', nperseg=n,
                          noverlap=n//2, detrend='constant', scaling='density', axis=-1)
    values = []
    for low, high in BANDS:
        mask = (freqs >= low) & (freqs <= high)
        if mask.sum() < 2:
            raise ValueError('Insufficient frequency bins.')
        power = np.trapezoid(density[...,mask], freqs[mask], axis=-1)
        values.append(np.log10(np.maximum(power, 1e-24)))
    return np.stack(values, axis=-1).reshape(len(X), -1).astype(np.float64)

def burg_coefficients(data, order=4):
    x = np.asarray(data, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Burg requires finite [signals, samples].')
    if not isinstance(order, (int,np.integer)) or not 1 <= order < x.shape[1]-1:
        raise ValueError('AR order must be positive and smaller than samples minus one.')
    x = x-x.mean(axis=-1,keepdims=True)
    scale = np.sqrt(np.mean(x*x,axis=-1,keepdims=True))
    x = x/np.maximum(scale,1e-15)
    forward,backward = x[:,1:].copy(),x[:,:-1].copy()
    coefficients = np.empty((len(x),0),dtype=np.float64)
    for lag in range(order):
        denominator = np.sum(forward*forward+backward*backward,axis=-1)
        reflection = np.divide(-2*np.sum(forward*backward,axis=-1),denominator,
            out=np.zeros(len(x)),where=denominator>1e-15)
        reflection = np.clip(reflection,-1+1e-7,1-1e-7)
        coefficients = np.concatenate([
            coefficients+reflection[:,None]*coefficients[:,::-1],reflection[:,None]],axis=1)
        updated_forward = forward+reflection[:,None]*backward
        updated_backward = backward+reflection[:,None]*forward
        forward,backward = updated_forward[:,1:],updated_backward[:,:-1]
    return -coefficients

def wavelet_statistics(coefficients):
    values = np.asarray(coefficients,dtype=np.float64)
    energy = values*values
    total = energy.sum(axis=-1,keepdims=True)
    probability = np.divide(energy,total,out=np.zeros_like(energy),where=total>1e-24)
    entropy = -np.sum(probability*np.log(np.maximum(probability,1e-300)),axis=-1)
    entropy = entropy/np.log(values.shape[-1]) if values.shape[-1]>1 else np.zeros_like(entropy)
    variance = np.log(np.maximum(np.var(values,axis=-1,ddof=0),1e-12))
    return np.clip(entropy,0.,1.),variance

def extract_trial_features(X,sfreq,ch_names,config):
    feature_set = config.get('feature_set','bandpower')
    rows = []
    def add(channel,window,family,component,start,stop):
        rows.append(dict(feature_index=len(rows),channel_index=int(channel),channel=ch_names[channel],
            window=int(window),family=family,component=component,
            start_time_mne=float(config['action_window_mne'][0]+start),
            stop_time_mne=float(config['action_window_mne'][0]+stop),
            feature_name=f'w{window:02d}__{ch_names[channel]}__{family}__{component}'))
    if feature_set=='bandpower':
        F = bandpower_features(X,sfreq)
        for channel in range(len(ch_names)):
            for lo,hi in BANDS:
                add(channel,0,'welch_log10_power',f'{lo:g}-{hi:g}Hz',0,X.shape[-1]/sfreq)
        return F,pd.DataFrame(rows),dict(feature_set=feature_set,units='volts',bands=BANDS,
            features=int(F.shape[1]),one_row_per_original_trial=True)
    if feature_set not in ['eight_window','combined']:
        raise ValueError('FEATURE_SET must be bandpower, eight_window, or combined.')
    import pywt
    n_windows = config.get('n_windows',8)
    wavelet = config.get('wavelet','db2')
    level = config.get('wavelet_level',3)
    ar_order = config.get('ar_order',4)
    target_rate = config.get('feature_sfreq',128.)
    lo,hi = config.get('wavelet_bandpass',[2.,45.])
    if n_windows!=8:
        raise ValueError('This reference implementation uses exactly eight windows.')
    if not 0 < lo < hi < min(sfreq,target_rate)/2:
        raise ValueError('Wavelet bandpass must be below both sampling Nyquist frequencies.')
    ratio = Fraction(float(target_rate)/float(sfreq)).limit_denominator(10000)
    samples = (X.shape[-1]*ratio.numerator+ratio.denominator-1)//ratio.denominator
    minimum_window = samples//n_windows
    maximum = pywt.dwt_max_level(minimum_window,wavelet)
    if not isinstance(level,(int,np.integer)) or not 1<=level<=maximum:
        raise ValueError(f'{wavelet} level {level} exceeds maximum {maximum} for {minimum_window}-sample windows.')
    if not isinstance(ar_order,(int,np.integer)) or not 1<=ar_order<minimum_window-1:
        raise ValueError('AR_ORDER is invalid for the window length.')
    microvolts = np.asarray(X,dtype=np.float32)*1e6
    sos = signal.butter(4,(lo,hi),btype='bandpass',fs=sfreq,output='sos')
    broadband = signal.sosfiltfilt(sos,microvolts,axis=-1).astype(np.float32)
    broadband = signal.resample_poly(broadband,ratio.numerator,ratio.denominator,axis=-1)
    components = [f'A{level}']+[f'D{i}' for i in range(level,0,-1)]
    descriptors = [('ar',f'lag{i}') for i in range(1,ar_order+1)]
    descriptors += [('wavelet_entropy',c) for c in components]
    descriptors += [('wavelet_log_variance',c) for c in components]
    windows = np.array_split(broadband,n_windows,axis=-1)
    matrices=[]
    offset=0
    for window,values in enumerate(windows,1):
        signals = values.reshape(-1,values.shape[-1])
        ar = burg_coefficients(signals,ar_order)
        coefficients = pywt.wavedec(signals.astype(np.float64),wavelet,
                                    mode='symmetric',level=level,axis=-1)
        summaries = [wavelet_statistics(c) for c in coefficients]
        entropy = np.stack([s[0] for s in summaries],axis=-1)
        variance = np.stack([s[1] for s in summaries],axis=-1)
        descriptor_values = np.concatenate([ar,entropy,variance],axis=-1)
        matrices.append(descriptor_values.reshape(len(X),-1))
        for channel in range(len(ch_names)):
            for family,component in descriptors:
                add(channel,window,family,component,offset/target_rate,(offset+values.shape[-1])/target_rate)
        offset += values.shape[-1]
    F = np.concatenate(matrices,axis=1)
    if feature_set=='combined':
        # Reference fusion: six filtered bands, whole epoch + three contiguous temporal bins.
        fusion_bands = ((2.,4.),(4.,8.),(8.,13.),(13.,20.),(20.,30.),(30.,45.))
        filtered = np.stack([signal.sosfiltfilt(signal.butter(4,band,btype='bandpass',fs=sfreq,output='sos'),
                                              microvolts,axis=-1).astype(np.float32) for band in fusion_bands],axis=1)
        edges = np.linspace(0,X.shape[-1],4).astype(int)
        bounds = [(0,X.shape[-1])]+list(zip(edges[:-1],edges[1:]))
        power = []
        for temporal,(start,stop) in enumerate(bounds):
            power.append(np.log(np.maximum(np.mean(filtered[...,start:stop]**2,axis=-1),1e-12)).reshape(len(X),-1))
            for low,high in fusion_bands:
                for channel in range(len(ch_names)):
                    add(channel,temporal,'filtered_log_power',f'{low:g}-{high:g}Hz',start/sfreq,stop/sfreq)
        F = np.concatenate([F,*power],axis=1)
    F = F.astype(np.float32)
    schema = pd.DataFrame(rows)
    if F.shape!=(len(X),len(schema)) or not np.isfinite(F).all() or schema.feature_name.duplicated().any():
        raise ValueError('Invalid descriptor values or schema.')
    info = dict(feature_set=feature_set,n_windows=n_windows,wavelet=wavelet,wavelet_level=level,
        wavelet_components=components,wavelet_mode='symmetric',boundary_coefficients='retained',
        ar_order=ar_order,ar_method='univariate_burg_demeaned',bandpass=[lo,hi],input_units='microvolts',
        input_sfreq=sfreq,feature_sfreq=target_rate,window_lengths=[w.shape[-1] for w in windows],
        entropy='normalized_negative_sum_p_log_p',variance='natural_log_population_variance_floor_1e-12',
        features=F.shape[1],one_row_per_original_trial=True,paper_exact_reproduction=False)
    return F,schema,info


In [ ]:
def safe_ratio(a, b):
    return a / np.maximum(np.abs(b), 1e-12)

def statistics25(x):
    """25 defined temporal descriptors, vectorized over leading dimensions."""
    x = np.asarray(x, np.float64)
    d = np.diff(x, axis=-1)
    dd = np.diff(d, axis=-1)
    mean, std = x.mean(axis=-1), x.std(axis=-1)
    absolute = np.abs(x)
    rms = np.sqrt(np.mean(x*x, axis=-1))
    variance = std**2
    mobility = np.sqrt(safe_ratio(np.var(d, axis=-1), variance))
    complexity = safe_ratio(np.sqrt(safe_ratio(np.var(dd, axis=-1), np.var(d, axis=-1))), mobility)
    # Central moments are explicitly zero for constant signals.
    standardized = (x - mean[..., None]) / np.maximum(std[..., None], 1e-12)
    values = [mean, std, variance, np.min(x, axis=-1), np.max(x, axis=-1),
              np.ptp(x, axis=-1), np.median(x, axis=-1),
              np.percentile(x, 75, axis=-1)-np.percentile(x, 25, axis=-1),
              absolute.mean(axis=-1), rms, np.sum(x*x, axis=-1),
              np.mean(standardized**3, axis=-1),
              np.where(std > 1e-12, np.mean(standardized**4, axis=-1)-3, 0),
              np.mean(np.abs(x-mean[..., None]), axis=-1),
              np.median(np.abs(x-np.median(x, axis=-1)[..., None]), axis=-1),
              np.mean(np.diff(np.signbit(x), axis=-1), axis=-1),
              np.sum(np.abs(d), axis=-1), np.mean(np.abs(d), axis=-1),
              np.mean(d*d, axis=-1), mobility, complexity,
              safe_ratio(np.max(absolute, axis=-1), rms),
              safe_ratio(rms, absolute.mean(axis=-1)),
              safe_ratio(np.max(absolute, axis=-1), absolute.mean(axis=-1)),
              safe_ratio(np.max(absolute, axis=-1), np.mean(np.sqrt(absolute), axis=-1)**2)]
    names = ["mean", "std", "variance", "min", "max", "range", "median", "iqr",
             "mean_absolute", "rms", "energy", "skewness", "excess_kurtosis",
             "mean_absolute_deviation", "median_absolute_deviation", "zero_crossing_rate",
             "waveform_length", "mean_abs_difference", "mean_squared_difference",
             "hjorth_mobility", "hjorth_complexity", "crest_factor", "shape_factor",
             "impulse_factor", "clearance_factor"]
    return np.stack(values, axis=-1), names

def frequency24(x, sfreq, high):
    freq, psd = signal.welch(x, fs=sfreq, nperseg=min(256, x.shape[-1]), axis=-1)
    keep = (freq >= 0.5) & (freq <= high)
    f, p = freq[keep], psd[..., keep]
    if len(f) < 3:
        raise ValueError("Too few frequency bins")
    prob = safe_ratio(p, p.sum(axis=-1)[..., None])
    center = (prob*f).sum(axis=-1)
    width = np.sqrt((prob*(f-center[..., None])**2).sum(axis=-1))
    cum = np.cumsum(prob, axis=-1)
    roll = lambda q: f[np.argmax(cum >= q, axis=-1)]
    entropy = -np.sum(prob*np.log(np.maximum(prob, 1e-300)), axis=-1)/np.log(len(f))
    flatness = safe_ratio(np.exp(np.log(np.maximum(p, 1e-300)).mean(axis=-1)), p.mean(axis=-1))
    peak = p.max(axis=-1)
    weighted_std = np.maximum(width, 1e-12)[..., None]
    centered_power = p-p.mean(axis=-1)[..., None]
    sd_power = np.maximum(p.std(axis=-1), 1e-12)[..., None]
    power_z = centered_power/sd_power
    values = [trapezoid(p, f, axis=-1), f[p.argmax(axis=-1)], center, width,
              roll(0.5), roll(0.85), roll(0.95), entropy, flatness,
              safe_ratio(peak, p.mean(axis=-1)),
              (prob*((f-center[..., None])/weighted_std)**3).sum(axis=-1),
              (prob*((f-center[..., None])/weighted_std)**4).sum(axis=-1),
              np.mean(power_z**3, axis=-1), np.mean(power_z**4, axis=-1)-3]
    names = ["total_power", "peak_frequency", "centroid", "bandwidth", "median_frequency",
             "rolloff85", "rolloff95", "entropy", "flatness", "crest",
             "frequency_skewness", "frequency_kurtosis", "psd_skewness", "psd_excess_kurtosis"]
    total = trapezoid(p, f, axis=-1)
    for name, lo, hi in [("delta", .5, 4), ("theta", 4, 8), ("alpha", 8, 13),
                         ("beta", 13, 30), ("gamma", 30, high)]:
        mask = (freq >= lo) & (freq < hi if hi < high else freq <= hi)
        power = trapezoid(psd[..., mask], freq[mask], axis=-1)
        values.extend([power, safe_ratio(power, total)])
        names.extend([f"{name}_power", f"{name}_relative_power"])
    return np.stack(values, axis=-1), names

def wavelet_features(x, wavelet, level):
    if pywt.dwt_max_level(x.shape[-1], pywt.Wavelet(wavelet).dec_len) < level:
        raise ValueError("Window is too short for the requested DWT level")
    coefficients = pywt.wavedec(x, wavelet, level=level, axis=-1, mode="symmetric")
    labels = [f"A{level}"] + [f"D{i}" for i in range(level, 0, -1)]
    values, names = [], []
    for band, c in zip(labels, coefficients):
        d = np.diff(c, axis=-1)
        std = c.std(axis=-1)
        z = (c-c.mean(axis=-1)[..., None])/np.maximum(std[..., None], 1e-12)
        probability = safe_ratio(c*c, np.sum(c*c, axis=-1)[..., None])
        # Geometric length of amplitude-normalized coefficients; definition explicit
        # because the paper's "fractal length" formula is unavailable.
        norm = safe_ratio(c-c.min(axis=-1)[..., None], np.ptp(c, axis=-1)[..., None])
        geometric_length = np.sum(np.sqrt(np.diff(norm, axis=-1)**2 + (1/max(c.shape[-1]-1, 1))**2), axis=-1)
        band_values = [c.mean(axis=-1), np.mean(np.abs(c), axis=-1), std, std**2,
                       np.sqrt(np.mean(c*c, axis=-1)), np.mean(c*c, axis=-1),
                       np.sum(c*c, axis=-1), np.sum(np.abs(d), axis=-1),
                       np.ptp(c, axis=-1), np.mean(z**3, axis=-1),
                       np.where(std > 1e-12, np.mean(z**4, axis=-1)-3, 0),
                       -np.sum(probability*np.log(np.maximum(probability, 1e-300)), axis=-1),
                       geometric_length]
        band_names = ["mean", "mean_absolute", "std", "variance", "rms", "mean_power",
                      "energy", "waveform_length", "range", "skewness", "excess_kurtosis",
                      "energy_entropy", "normalized_geometric_length"]
        values.extend(band_values)
        names.extend([f"{band}_{n}" for n in band_names])
    return np.stack(values, axis=-1), names

def documented_features(trial, cfg):
    """50 TD + 24 FD + 78 TFD = 152/channel; deliberately not padded to 191."""
    td, td_names = statistics25(trial)
    env, env_names = statistics25(np.abs(signal.hilbert(trial, axis=-1)))
    fd, fd_names = frequency24(trial, cfg.expected_sfreq, cfg.spectral_high)
    tf, tf_names = wavelet_features(trial, cfg.wavelet, cfg.wavelet_level)
    matrix = np.concatenate([td, env, fd, tf], axis=-1)
    names = ([f"TD_signal_{n}" for n in td_names] + [f"TD_envelope_{n}" for n in env_names]
             + [f"FD_{n}" for n in fd_names] + [f"TFD_{n}" for n in tf_names])
    if matrix.shape[-1] != 152 or len(set(names)) != 152:
        raise AssertionError("Feature schema changed")
    if not np.isfinite(matrix).all():
        raise ValueError("Non-finite features; do not silently replace numerical failures")
    return matrix, names


## 4. Audited loader and feature cache

Only condition1 (INNER SPEECH) is retained. Event labels and timestamps are checked against FIF epochs; original retained rows, subject, session and trial IDs are preserved. The derivative epoch clock is cropped with `epochs.times` to `[0.5,3.0)`, rather than guessed sample indices. For the audited 256 Hz files, this yields 640 samples per trial. Source hashes, rates, channels and feature schema are saved.

The input reader deliberately requires all128 source channels. Channel reduction happens afterward, through a validated schema mask. Features are cached and shared across budgets and boundaries; ranking/model caches also include exact channel membership, target labels, trial rows, parameters and code versions.


In [ ]:
def environment_info():
    result = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                  scipy=scipy.__version__, sklearn=sklearn.__version__)
    for name in ('mne', 'PyWavelets', 'lightgbm', 'vmdpy'):
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = 'not installed'
    return result

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.tmp')
    with temp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    temp.replace(path)

def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.csv.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)

def boundary_label(boundary):
    return PAPER_LABEL if boundary == 'PAPER_ORDER' else SAFE_LABEL

def discover_data_root(requested):
    if requested is not None:
        root = Path(requested)
        if not root.exists():
            raise FileNotFoundError(f'DATA_ROOT does not exist: {root}')
        return root
    base = Path('/kaggle/input')
    parents = {p.parent for p in base.rglob('sub-*_ses-*_eeg-epo.fif')} if base.exists() else set()
    if not parents:
        raise FileNotFoundError('No processed FIF files discovered. Set CONFIG["data_root"].')
    # Select the enclosing derivatives directory, never silently combine dataset copies.
    roots = {next((q for q in [p, *p.parents] if q.name == 'derivatives'), p) for p in parents}
    if len(roots) != 1:
        raise ValueError(f'Multiple dataset roots: {sorted(map(str, roots))}. Set data_root explicitly.')
    return roots.pop()

def discover_subjects(root):
    return sorted({int(re.match(r'sub-(\d+)_ses-', p.name)[1])
                   for p in Path(root).rglob('sub-*_ses-*_eeg-epo.fif')})

def vmd_backend():
    try:
        from vmdpy import VMD
        return VMD, 'vmdpy'
    except ImportError:
        try:
            from sktime.libs.vmdpy import VMD
            return VMD, 'sktime.libs.vmdpy'
        except ImportError:
            return None, 'unavailable'

def robust_z(x):
    med = np.median(x)
    return np.abs(x-med) / max(1.4826*np.median(np.abs(x-med)), 1e-12)

def apply_vmd(X, sfreq, names, subject, config):
    mode = config['vmd_mode']
    if mode == 'off':
        return X, dict(requested=mode, effective='off', backend='unused', applications=[])
    VMD, backend = vmd_backend()
    if VMD is None:
        print(f'Subject {subject:02d}: VMD requested but no installed backend; SKIPPED. No pip/internet used.', flush=True)
        return X, dict(requested=mode, effective='skipped_missing_backend', backend=backend, applications=[])
    cleaned = np.asarray(X, dtype=np.float64).copy()
    report = dict(requested=mode, effective=mode, backend=backend, applications=[],
                  approximation='Manual choices or robust per-trial channel mean/log-std detector; threshold and VMD parameters are assumptions.')
    manual = config['vmd_manual_channels'].get(subject, config['vmd_manual_channels'].get(str(subject), []))
    if any(name not in names for name in manual):
        raise ValueError(f'Unknown manual VMD channels: {set(manual)-set(names)}')
    for row, trial in enumerate(cleaned):
        if mode == 'manual':
            mask = np.array([name in manual for name in names])
        else:
            # Label-free, trial-local across-channel detector, no held-out-distribution fit.
            uv = trial*1e6
            mask = ((robust_z(uv.mean(axis=1)) > config['vmd_robust_z']) |
                    (robust_z(np.log(np.maximum(uv.std(axis=1), 1e-12))) > config['vmd_robust_z']))
        for col in np.flatnonzero(mask):
            x = trial[col]*1e6
            if np.std(x) < 1e-12:
                continue
            scale = max(float(np.sqrt(np.mean(x*x))), 1e-12)
            padded = np.pad(x/scale, (0, len(x)%2), mode='edge')
            p = config['vmd_parameters']
            modes, _, omega = VMD(padded, p['alpha'], p['tau'], 6, p['dc'], p['init'], p['tol'])
            if len(modes) != 6 or omega.ndim != 2:
                raise ValueError('Unexpected VMD output.')
            lowest = int(np.argmin(omega[-1]))
            clean = modes[np.arange(6) != lowest].sum(axis=0)[:len(x)] * scale / 1e6
            if not np.isfinite(clean).all():
                raise ValueError('Nonfinite VMD output.')
            trial[col] = clean
            report['applications'].append(dict(trial_row=row, channel=names[col], removed_mode=lowest,
                                               removed_hz=float(omega[-1, lowest]*sfreq)))
        if row == 0 or (row+1)%20 == 0:
            print(f'  VMD trial {row+1}/{len(X)}', flush=True)
    return cleaned.astype(np.float32), report

def feature_matrix(X, sfreq, names, mode, config):
    if mode == 'current':
        p = dict(config['current_features'], action_window_mne=config['action_window_mne'])
        F, schema, info = extract_trial_features(X, sfreq, names, p)
        schema = schema.copy()
        schema['original_feature_name'] = schema.feature_name
        schema['feature_name'] = schema.apply(lambda r: f'{r.channel}__w{r.window:02d}__{r.family}__{r.component}', axis=1)
        info['note'] = 'Current feature values and ordering reused exactly; names reformatted only.'
    else:
        cfg = SimpleNamespace(expected_sfreq=sfreq, **config['paper_inspired_features'])
        if cfg.wavelet_level != 5:
            raise ValueError('The existing documented_152 schema requires wavelet_level=5.')
        if not 30 < cfg.spectral_high <= sfreq/2:
            raise ValueError('spectral_high must be >30 and <= Nyquist.')
        # Existing approximation uses microvolts; explicit unit avoids scale-dependent numerical floors.
        values, feature_names = documented_features(X.astype(np.float64)*1e6, cfg)
        F = values.reshape(len(X), -1).astype(np.float32)
        schema = pd.DataFrame([dict(feature_index=i*len(feature_names)+j, channel_index=i,
                                   channel=channel, family=name.split('_')[0], component=name,
                                   feature_name=f'{channel}__{name}')
                              for i, channel in enumerate(names) for j, name in enumerate(feature_names)])
        info = dict(feature_mode=mode, features_per_channel=len(feature_names), units='microvolts',
                    wavelet=cfg.wavelet, wavelet_level=cfg.wavelet_level, spectral_high=cfg.spectral_high,
                    counts=dict(TD=50, FD=24, TFD=78), paper_exact_reproduction=False,
                    geometric_length='Amplitude-normalized polygonal length; not verified as the paper fractal-length formula.')
    if F.shape[1] != len(schema) or schema.feature_name.duplicated().any() or not np.isfinite(F).all():
        raise ValueError('Invalid feature schema or nonfinite values.')
    counts = schema.groupby('channel').size()
    if counts.nunique() != 1:
        raise ValueError('Feature counts differ across channels.')
    print(f'{"Paper-inspired feature set" if mode == "paper_inspired" else "Current feature set"}: '
          f'{int(counts.iloc[0])} features/channel; {F.shape[1]} total. Not an exact 191-feature replication.', flush=True)
    return F, schema, info

def prepare_subject_replication(subject, config):
    import mne
    root = Path(config['data_root'])
    files = sorted(root.rglob(f'sub-{subject:02d}_ses-*_eeg-epo.fif'))
    if config['sessions'] is not None:
        files = [p for p in files if int(re.search(r'_ses-(\d+)_', p.name)[1]) in config['sessions']]
    if not files:
        raise FileNotFoundError(f'No requested sessions for subject {subject}.')
    session_ids = [int(re.search(r'_ses-(\d+)_', p.name)[1]) for p in files]
    if len(set(session_ids)) != len(session_ids):
        raise ValueError('Duplicate subject/session FIF files; choose one dataset version.')
    if config['sessions'] is not None and set(session_ids) != set(config['sessions']):
        raise ValueError('Some requested sessions are missing.')
    paths = [q for p in files for q in (p, p.with_name(p.name.replace('_eeg-epo.fif','_events.dat')))]
    sources = [dict(path=str(p), sha256=file_sha256(p)) for p in paths]
    feature_config = {k: config[k] for k in ('action_window_mne', 'feature_mode', 'current_features',
                      'paper_inspired_features', 'vmd_mode', 'vmd_manual_channels', 'vmd_robust_z', 'vmd_parameters')}
    _, backend = vmd_backend() if config['vmd_mode'] != 'off' else (None, 'unused')
    key = digest(dict(subject=subject, sources=sources, feature_config=feature_config,
                      engine=ENGINE_ID, versions=environment_info(), vmd_backend=backend))
    cache = Path(config['output_root'])/'feature_cache'/key
    if config['resume'] and (cache/'manifest.json').exists():
        manifest = json.loads((cache/'manifest.json').read_text())
        F = np.load(cache/'features.npz')['F']
        metadata = pd.read_csv(cache/'metadata.csv')
        schema = pd.read_csv(cache/'schema.csv')
        if array_digest(F) != manifest['F_sha256'] or len(F) != len(metadata):
            raise ValueError('Feature cache integrity check failed.')
        print(f'Subject {subject:02d}: feature cache reused ({F.shape}); VMD: {manifest["vmd_effective"]}', flush=True)
        return dict(subject=subject, F=F, y=metadata.class_id.to_numpy(dtype=int), metadata=metadata,
                    schema=schema, manifest=manifest, feature_key=key)
    arrays, records, provenance, vmd_reports = [], [], [], []
    reference = None
    for path, session in zip(files, session_ids):
        event_path = path.with_name(path.name.replace('_eeg-epo.fif','_events.dat'))
        # Only the trusted dataset's event pickle is used, as in the audited existing loader.
        raw = np.asarray(pd.read_pickle(event_path))
        epochs = mne.read_epochs(path, preload=False, proj=False, verbose='ERROR')
        if raw.shape != (len(epochs),4) or not np.isfinite(raw).all() or not np.equal(raw,np.floor(raw)).all():
            raise ValueError(f'Invalid event dimensions or values: {event_path}')
        events = raw.astype(np.int64)
        if not np.array_equal(events[:,0],epochs.events[:,0]) or not np.array_equal(events[:,1]+31,epochs.events[:,2]):
            raise ValueError('FIF/event alignment failed.')
        if not np.isin(events[:,1],LABELS).all():
            raise ValueError('Unknown direction codes.')
        original_rows = np.asarray(epochs.selection)
        inner = np.flatnonzero(events[:,2] == 1)
        if not len(inner):
            raise ValueError('No INNER SPEECH trials in a requested session.')
        epochs = epochs[inner].load_data()
        epochs.pick(mne.pick_types(epochs.info,eeg=True,exclude=[]))
        names, sfreq = list(epochs.ch_names), float(epochs.info['sfreq'])
        if len(names) != 128:
            raise ValueError('This replication keeps all 128 EEG channels.')
        start, stop = config['action_window_mne']
        if start < epochs.times[0] or stop > epochs.times[-1]+1/sfreq+1e-9:
            raise ValueError('Requested action interval is outside the FIF epoch.')
        mask = (epochs.times >= start) & (epochs.times < stop)
        times = epochs.times[mask]
        if not len(times):
            raise ValueError('Empty action interval.')
        if reference is not None and (names != reference[0] or sfreq != reference[1] or not np.array_equal(times,reference[2])):
            raise ValueError('Sessions have different channel orders, rates or action times.')
        reference = names,sfreq,times
        X = epochs.get_data(copy=False)[:,:,mask].astype(np.float32)
        if not np.isfinite(X).all():
            raise ValueError('Nonfinite EEG.')
        X, report = apply_vmd(X,sfreq,names,subject,config)
        vmd_reports.append(dict(session=session, **report))
        arrays.append(X)
        for row, event in zip(original_rows[inner], events[inner]):
            records.append(dict(subject=subject,session=session,trial_index=int(row),cue_sample_raw=int(event[0]),
                                class_id=int(event[1]),condition='INNER',
                                trial_id=f'sub-{subject:02d}_ses-{session:02d}_row-{row:04d}'))
        provenance.append(dict(file=str(path),session=session,shape=list(X.shape),sfreq=sfreq,
                               first_time=float(times[0]),last_time=float(times[-1]),X_sha256=array_digest(X),
                               highpass=float(epochs.info['highpass']),lowpass=float(epochs.info['lowpass'])))
        print(f'Subject {subject:02d} session {session:02d}: X={X.shape}, sfreq={sfreq:g} Hz, INNER only.', flush=True)
    X = np.concatenate(arrays)
    metadata = pd.DataFrame(records)
    if metadata.trial_id.duplicated().any() or metadata.subject.nunique() != 1:
        raise ValueError('Trial IDs or subject separation failed.')
    F,schema,feature_info = feature_matrix(X,sfreq,names,config['feature_mode'],config)
    manifest = dict(subject=subject,pooled_shape=list(X.shape),sfreq=sfreq,channels=names,sources=sources,
                    provenance=provenance,feature_info=feature_info,feature_config=feature_config,
                    F_sha256=array_digest(F),vmd=vmd_reports,
                    vmd_effective=sorted({r['effective'] for r in vmd_reports}),feature_key=key,
                    action_interval_note='FIF epoch clock, audited cue-relative [0.5,3.0); original protocol action [1.0,3.5).')
    atomic_npz(cache/'features.npz',F=F)
    atomic_csv(metadata,cache/'metadata.csv')
    atomic_csv(schema,cache/'schema.csv')
    save_json(cache/'manifest.json',manifest)
    return dict(subject=subject,F=F,y=metadata.class_id.to_numpy(dtype=int),metadata=metadata,
                schema=schema,manifest=manifest,feature_key=key)

def dataset_summary(datasets, output_dir):
    rows = []
    for data in datasets.values():
        for session, meta in data['metadata'].groupby('session'):
            counts = meta.class_id.value_counts()
            rows.append(dict(subject=data['subject'],session=int(session),trials=len(meta),
                             trials_in_subject=len(data['y']),EEG_shape=str([len(meta),128,data['manifest']['pooled_shape'][2]]),
                             sfreq=data['manifest']['sfreq'],channels=128,
                             **{f'class_{i}':int(counts.get(i,0)) for i in LABELS}))
    table = pd.DataFrame(rows)
    atomic_csv(table,Path(output_dir)/'dataset_summary.csv')
    print('Subjects:', sorted(datasets), '\nSessions per subject:', table.groupby('subject').session.apply(list).to_dict())
    return table


## 5. Reused selection and evaluation engine

MRMR considers all nonconstant candidates in the chosen channel set, with no ANOVA prefilter and no feature-by-feature correlation matrix. It uses relevance divided by mean absolute Pearson redundancy. Constant eligibility follows the existing std > 1e-12 criterion in declared feature units. K is never silently reduced.

All classifier scalers use outer train only, even in PAPER_ORDER; the deliberate boundary difference is supervised selection. NESTED_SAFE refits selection on each outer train and, if SVM tuning is enabled, on each inner train. The optional five LR ensemble bootstraps train rows only, fits one scaler per bootstrap and averages aligned probabilities. Exactly one OOF prediction per original trial is checked. Checkpoints include selected feature names and fit/test/bootstrap rows.


In [ ]:
class MRMRQuotient(TransformerMixin, BaseEstimator):
    """Paper Eq.1 form: relevance / mean absolute correlation to selected features.

    MI and absolute Pearson correlation are explicit implementation choices.
    All candidates are considered; no relevance prefilter and no p-by-p matrix.
    """
    def __init__(self, k=590, relevance="mi", corr_floor=1e-6, seed=42):
        self.k, self.relevance, self.corr_floor, self.seed = k, relevance, corr_floor, seed

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        if len(X) != len(y) or not np.isfinite(X).all():
            raise ValueError("Invalid MRMR training input")
        self.n_features_in_ = X.shape[1]
        sd = X.std(axis=0)
        available_columns = np.flatnonzero(sd > 1e-12)
        if not len(available_columns):
            raise ValueError("All feature columns are constant")
        Z = (X[:, available_columns] - X[:, available_columns].mean(axis=0)) / sd[available_columns]
        if self.relevance == "mi":
            rel = mutual_info_classif(Z, y, random_state=self.seed)
        elif self.relevance == "f_stat":
            rel = f_classif(Z, y)[0]
        else:
            raise ValueError("Unknown MRMR relevance estimator")
        rel = np.nan_to_num(rel, nan=0, posinf=1e12, neginf=0)
        remaining = np.ones(len(rel), dtype=bool)
        redundancy_sum = np.zeros(len(rel), dtype=np.float64)
        chosen = []
        for step in range(min(int(self.k), len(rel))):
            score = rel.copy() if step == 0 else rel / np.maximum(redundancy_sum / step, self.corr_floor)
            score[~remaining] = -np.inf
            nxt = int(np.argmax(score))
            chosen.append(nxt)
            remaining[nxt] = False
            redundancy_sum += np.abs(Z.T @ Z[:, nxt] / len(Z))
        self.selected_indices_ = available_columns[np.asarray(chosen)]
        self.relevance_values_ = rel
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.selected_indices_]


In [ ]:
def mutual_info_classif(X, y, random_state=42):
    """Same sklearn estimator, independent columns batched to expose progress."""
    parts = []
    for start in range(0, X.shape[1], 2048):
        stop = min(start+2048, X.shape[1])
        print(f'    MI relevance: features {start+1}–{stop}/{X.shape[1]}', flush=True)
        # Keep one RNG stream, as in a single call to sklearn.
        if start == 0:
            rng = np.random.RandomState(random_state)
        parts.append(_sklearn_mutual_info_classif(X[:,start:stop],y,random_state=rng))
    return np.concatenate(parts)

def cv_plan(data, config, scope='pooled_subject'):
    n = len(data['y'])
    units = [np.arange(n)] if scope == 'pooled_subject' else [g.index.to_numpy() for _,g in data['metadata'].groupby('session')]
    plan, visits = [], np.zeros(n,dtype=int)
    for unit in units:
        y = data['y'][unit]
        counts = np.bincount(y,minlength=4)
        if counts.min() < config['outer_folds']:
            raise ValueError(f'All four classes require >= {config["outer_folds"]} trials for this scope; counts={counts}. No silent fold reduction.')
        cv = StratifiedKFold(config['outer_folds'],shuffle=True,random_state=config['seed'])
        session = 0 if scope == 'pooled_subject' else int(data['metadata'].iloc[unit[0]].session)
        for fold,(a,b) in enumerate(cv.split(np.zeros(len(unit)),y),1):
            train,test = unit[a],unit[b]
            assert not np.intersect1d(train,test).size
            assert data['metadata'].iloc[train].subject.nunique() == 1
            if scope != 'pooled_subject':
                assert data['metadata'].iloc[train].session.nunique() == 1
            visits[test] += 1
            plan.append(dict(fold=fold,unit_session=session,train=train.tolist(),test=test.tolist(),selection_unit=unit.tolist()))
    assert np.all(visits==1), 'Exactly one held-out prediction per trial is required.'
    return plan

def ranking(data, fit_rows, selector, k_max, config, root):
    fit_rows = np.asarray(fit_rows,dtype=int)
    y = data['y'][fit_rows]
    key = digest(dict(feature_key=data['feature_key'],F=data['manifest']['F_sha256'],rows=fit_rows.tolist(),
                      y=array_digest(y),selector=selector,relevance=config['mrmr_relevance'],
                      corr_floor=config['mrmr_corr_floor'],seed=config['seed'],engine=ENGINE_ID,
                      versions=environment_info()))
    directory = Path(root)/'rank_cache'/key
    if config['resume'] and (directory/'ranking.npz').exists():
        with np.load(directory/'ranking.npz') as cached:
            indices = cached['indices']
        if len(indices) >= k_max:
            return indices[:k_max]
    F = data['F'][fit_rows].astype(np.float64)
    # Reused selector normalization is fitted at precisely fit_rows.
    eligible = np.flatnonzero(F.std(axis=0)>1e-12)
    if k_max > len(eligible):
        raise ValueError(f'K={k_max} exceeds {len(eligible)} eligible features; no silent clipping.')
    began = perf_counter()
    print(f'  Ranking {selector.upper()}: {len(fit_rows)} fit trials, {len(eligible)} candidates, K={k_max}; '
          f'relevance={config["mrmr_relevance"] if selector=="mrmr" else "f_stat"}', flush=True)
    with threadpool_limits(limits=config['blas_threads']):
        if selector == 'anova':
            values = np.nan_to_num(f_classif(F[:,eligible],y)[0],nan=0,posinf=1e12,neginf=0)
            indices = eligible[np.argsort(-values,kind='stable')[:k_max]]
        elif selector == 'mrmr':
            ranker = MRMRQuotient(k_max,config['mrmr_relevance'],config['mrmr_corr_floor'],config['seed'])
            ranker.fit(F,y)
            indices = ranker.selected_indices_
        else:
            raise ValueError('Unknown selector.')
    if len(indices)!=k_max or len(set(indices))!=k_max:
        raise ValueError('Selector did not return the requested unique feature count.')
    atomic_npz(directory/'ranking.npz',indices=indices)
    save_json(directory/'audit.json',dict(fit_rows=fit_rows.tolist(),fit_trial_ids=data['metadata'].iloc[fit_rows].trial_id.tolist(),
                                       labels_sha256=array_digest(y),selector=selector,k_max=k_max,eligible_features=len(eligible)))
    print(f'  Ranking done in {(perf_counter()-began)/60:.2f} minutes; prefixes reused for smaller K.', flush=True)
    return indices

class StratifiedLR5(ClassifierMixin, BaseEstimator):
    """Five configurable LR pipelines, stratified training-only bootstrap, mean probability vote."""
    def __init__(self, members=None, seed=42, max_iter=5000):
        self.members,self.seed,self.max_iter = members,seed,max_iter

    def fit(self, X, y):
        if self.members is None or len(self.members)!=5:
            raise ValueError('Exactly five LR member configurations are required.')
        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        self.models_,self.bootstrap_indices_ = [],[]
        rng = np.random.default_rng(self.seed)
        for i, spec in enumerate(self.members):
            indices = np.concatenate([rng.choice(np.flatnonzero(y==c),np.sum(y==c),replace=True) for c in self.classes_])
            rng.shuffle(indices)
            params = dict(spec,max_iter=self.max_iter,random_state=self.seed+i)
            # Explicit OVR supports liblinear in multiclass tasks across sklearn versions.
            model = Pipeline([('scale',StandardScaler()),('lr',OneVsRestClassifier(LogisticRegression(**params),n_jobs=1))])
            model.fit(X[indices],y[indices])
            assert np.array_equal(model.classes_,self.classes_)
            self.models_.append(model)
            self.bootstrap_indices_.append(indices)
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models_],axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X),axis=1)]

def make_model(name, config, override=None):
    seed = config['seed']
    params = dict(config['models'].get(name,{}))
    params.update(override or {})
    if name == 'svm':
        estimator,scale = SVC(**params),True
    elif name == 'lda':
        estimator,scale = LinearDiscriminantAnalysis(**params),True
    elif name == 'etc':
        estimator,scale = ExtraTreesClassifier(**params,random_state=seed,n_jobs=config['n_jobs']),False
    elif name == 'rf':
        estimator,scale = RandomForestClassifier(**params,random_state=seed,n_jobs=config['n_jobs']),False
    elif name == 'lr':
        estimator,scale = LogisticRegression(**params,random_state=seed),True
    elif name == 'lgbm':
        try:
            from lightgbm import LGBMClassifier
        except ImportError as exc:
            raise ImportError('LightGBM is unavailable; no substitute result is labeled LightGBM.') from exc
        estimator,scale = LGBMClassifier(**params,random_state=seed,n_jobs=config['n_jobs'],verbosity=-1),False
    elif name == 'lr5':
        return StratifiedLR5(config['lr5_members'],seed,config['lr5_max_iter'])
    else:
        raise ValueError(f'Unknown classifier: {name}')
    return Pipeline([('scale',StandardScaler() if scale else 'passthrough'),('model',estimator)])

def metrics(y, pred):
    rec = recall_score(y,pred,labels=LABELS,average=None,zero_division=0)
    return dict(accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                macro_f1=float(f1_score(y,pred,labels=LABELS,average='macro',zero_division=0)),
                **{f'recall_{i}':float(rec[i]) for i in LABELS})

def choose_inner(data, train, boundary, fixed_columns, selector, counts, config, root):
    counts = sorted(set(map(int,counts)))
    n_inner = config['inner_folds']
    if np.bincount(data['y'][train],minlength=4).min()<n_inner:
        raise ValueError('Insufficient outer-training examples for configured inner CV.')
    candidates = list(ParameterGrid(config['svm_inner_grid']))
    scores = np.zeros((len(counts),len(candidates),n_inner))
    cv = StratifiedKFold(n_inner,shuffle=True,random_state=config['seed'])
    audits = []
    for split,(a,b) in enumerate(cv.split(np.zeros(len(train)),data['y'][train])):
        inner_train,inner_valid = train[a],train[b]
        if boundary == 'NESTED_SAFE':
            columns = ranking(data,inner_train,selector,max(counts),config,root)
            selection_rows = inner_train
        else:
            columns = fixed_columns
            selection_rows = None # Pre-CV selection intentionally exposed these labels.
        assert not np.intersect1d(inner_train,inner_valid).size
        audits.append(dict(inner_fold=split+1,train=inner_train.tolist(),validation=inner_valid.tolist(),
                           selection_rows=None if selection_rows is None else selection_rows.tolist()))
        for ki,k in enumerate(counts):
            for pi,params in enumerate(candidates):
                model = make_model('svm',config,params)
                with threadpool_limits(limits=config['blas_threads']):
                    model.fit(data['F'][inner_train][:,columns[:k]],data['y'][inner_train])
                    p = model.predict(data['F'][inner_valid][:,columns[:k]])
                scores[ki,pi,split] = balanced_accuracy_score(data['y'][inner_valid],p)
    ki,pi = np.unravel_index(np.argmax(scores.mean(axis=2)),scores.shape[:2])
    return counts[ki],candidates[pi],dict(inner_splits=audits,candidates=candidates,k_values=counts,
                                       mean_balanced_accuracy=scores.mean(axis=2).tolist(),selected_from='inner validation only')

def evaluate(data, spec, boundary, config, root, scope='pooled_subject'):
    selector, requested_k, classifier = spec['selector'],spec['k'],spec['classifier']
    plan = cv_plan(data,config,scope)
    setting = dict(subject=data['subject'],feature_mode=config['feature_mode'],scope=scope,boundary=boundary,
                   selection_boundary=boundary_label(boundary),**spec)
    result_key = digest(dict(setting=setting,feature_key=data['feature_key'],plan=plan,
                             evaluation_config={k:config[k] for k in ('seed','outer_folds','inner_folds','mrmr_relevance',
                             'mrmr_corr_floor','models','lr5_members','lr5_max_iter','tune_svm','svm_inner_grid','k_values')},
                             engine=ENGINE_ID,versions=environment_info()))
    directory = Path(root)/'runs'/result_key
    directory.mkdir(parents=True,exist_ok=True)
    k_max = max(config['k_values']) if requested_k == 'inner' else int(requested_k)
    n = len(data['y'])
    predictions = np.full(n,-1,dtype=int)
    probabilities = np.full((n,4),np.nan)
    outer_fold,unit_session,visits = np.zeros(n,dtype=int),np.zeros(n,dtype=int),np.zeros(n,dtype=int)
    selections,fold_rows,audits = [],[],[]
    save_json(directory/'setting.json',dict(**setting,result_id=result_key,feature_key=data['feature_key']))
    save_json(directory/'plan.json',plan)
    for fold in plan:
        train,test = np.asarray(fold['train']),np.asarray(fold['test'])
        selection_rows = np.asarray(fold['selection_unit']) if boundary=='PAPER_ORDER' else train
        token = f"ses-{fold['unit_session']:02d}_fold-{fold['fold']:02d}"
        checkpoint = directory/f'{token}.npz'
        audit_path = directory/f'{token}.json'
        if config['resume'] and checkpoint.exists() and audit_path.exists():
            with np.load(checkpoint) as saved:
                pred,cols,prob = saved['pred'],saved['columns'],saved['probabilities']
            audit = json.loads(audit_path.read_text())
            if audit['test_rows'] != test.tolist() or audit['train_rows'] != train.tolist() or len(pred)!=len(test):
                raise ValueError('Checkpoint split mismatch.')
            chosen_k = audit['chosen_k']
            print(f"  Subject {data['subject']:02d}, fold {fold['fold']:02d}: resumed {boundary_label(boundary)}",flush=True)
        else:
            start = perf_counter()
            print(f"Subject {data['subject']:02d}, {scope}, {selector.upper()}, K={requested_k}, {classifier.upper()}, "
                  f"{boundary_label(boundary)}, fold {fold['fold']}/{config['outer_folds']} (session unit {fold['unit_session']}).",flush=True)
            cols = ranking(data,selection_rows,selector,k_max,config,root)
            chosen_k,params,inner_audit = k_max,{},None
            if classifier=='svm' and (config['tune_svm'] or requested_k=='inner'):
                counts = config['k_values'] if requested_k=='inner' else [k_max]
                chosen_k,params,inner_audit = choose_inner(data,train,boundary,cols,selector,counts,config,root)
            cols = cols[:chosen_k]
            if boundary=='NESTED_SAFE':
                assert not np.intersect1d(selection_rows,test).size
            model = make_model(classifier,config,params)
            with threadpool_limits(limits=config['blas_threads']):
                model.fit(data['F'][train][:,cols],data['y'][train])
                pred = model.predict(data['F'][test][:,cols])
                train_pred = model.predict(data['F'][train][:,cols])
                prob = model.predict_proba(data['F'][test][:,cols]) if hasattr(model,'predict_proba') else np.empty((len(test),0))
            bootstrap = []
            if classifier=='lr5':
                bootstrap = [train[x].tolist() for x in model.bootstrap_indices_]
                assert all(set(x).issubset(set(train)) and not set(x).intersection(test) for x in bootstrap)
            audit = dict(train_rows=train.tolist(),test_rows=test.tolist(),selection_fit_rows=selection_rows.tolist(),
                         scaler_fit_rows=train.tolist(),bootstrap_global_rows=bootstrap,
                         bootstrap_scalers_fit_on='each bootstrap sample' if classifier=='lr5' else 'outer train',
                         chosen_k=int(chosen_k),model_overrides=params,inner_selection=inner_audit,
                         train_scores=metrics(data['y'][train],train_pred),elapsed_seconds=perf_counter()-start,
                         selection_boundary=boundary_label(boundary))
            atomic_npz(checkpoint,pred=pred,columns=cols,probabilities=prob)
            save_json(audit_path,audit)
            print(f"  Held-out BA={balanced_accuracy_score(data['y'][test],pred):.3f}; fold finished in {audit['elapsed_seconds']/60:.2f} minutes.",flush=True)
        predictions[test] = pred
        if prob.shape == (len(test),4):
            probabilities[test] = prob
        outer_fold[test] = fold['fold']
        unit_session[test] = fold['unit_session']
        visits[test] += 1
        if boundary=='NESTED_SAFE':
            assert not set(audit['selection_fit_rows']).intersection(test)
        assert set(audit['scaler_fit_rows']).issubset(train)
        assert all(set(x).issubset(train) for x in audit['bootstrap_global_rows'])
        fold_rows.append(dict(**setting,fold=fold['fold'],unit_session=fold['unit_session'],chosen_k=chosen_k,
                              train_accuracy=audit['train_scores']['accuracy'],train_balanced_accuracy=audit['train_scores']['balanced_accuracy'],
                              **metrics(data['y'][test],pred)))
        selections.extend([dict(result_id=result_key,fold=fold['fold'],unit_session=fold['unit_session'],rank=i+1,
                                feature_index=int(col),feature_name=data['schema'].iloc[col].feature_name,
                                channel=data['schema'].iloc[col].channel,**setting) for i,col in enumerate(cols)])
        audits.append(audit)
    assert np.all(visits==1) and np.isin(predictions,LABELS).all()
    assert data['metadata'].subject.nunique()==1
    oof = data['metadata'].copy()
    oof['y_true'],oof['y_pred'],oof['outer_fold'],oof['unit_session'] = data['y'],predictions,outer_fold,unit_session
    if np.isfinite(probabilities).all():
        for label in LABELS:
            oof[f'probability_{label}'] = probabilities[:,label]
    for name,value in setting.items():
        oof[name]=value
    row = dict(**setting,result_id=result_key,n_trials=n,OOF_predictions_per_trial=1,
               vmd_effective=str(data['manifest']['vmd_effective']),**metrics(data['y'],predictions))
    selection_frame = pd.DataFrame(selections)
    atomic_csv(oof,directory/'oof.csv')
    atomic_csv(selection_frame,directory/'selected_features.csv')
    atomic_csv(pd.DataFrame(fold_rows),directory/'fold_scores.csv')
    atomic_csv(pd.DataFrame(confusion_matrix(data['y'],predictions,labels=LABELS),columns=[f'pred_{i}' for i in LABELS])
               .assign(true_class=LABELS),directory/'confusion.csv')
    save_json(directory/'result.json',row)
    return dict(row=row,oof=oof,selections=selection_frame,fold_scores=pd.DataFrame(fold_rows),directory=directory,audits=audits)


## 6. Fixed spatial configurations and paired channel driver

The geometry rule starts near the highest scalp point, then repeatedly selects the electrode farthest from previously chosen electrodes. It runs once in canonical BioSemi 128 order without EEG values or class labels, giving nested16 ⊂ 32 ⊂ 64 ⊂ 128 sets. Membership and order are frozen and exported. Geometry guarantees a reproducible spread, not mathematically optimal anatomical coverage.

The driver restricts both feature values and schema before ANOVA/MRMR. Ranking sees only candidates from that budget. Source feature indices are preserved for later analysis. Reports also distinguish available channels from the channels actually represented among selected features.


In [ ]:
"""Fixed spatial channel sets and paired evaluation-boundary comparisons."""
from copy import deepcopy
CHANNEL_EXTENSION_VERSION='channel-reduction-dual-1.0'
CHANNEL_MODEL_NAMES={'svm','lda','etc','rf','lr','lgbm','lr5'}
CHANNEL_BOUNDARY_COLORS={'PAPER_ORDER':'#d95f02','NESTED_SAFE':'#1b9e77'}


In [ ]:
def validate_channel_config(config):
    for key in ('subjects','channel_counts','boundaries','selectors','classifiers'):
        value=config[key]
        if key=='subjects' and value is None:
            continue
        if not value or len(value)!=len(set(value)):
            raise ValueError(f'{key} must be a nonempty list of unique values, or subjects=None.')
    if 128 not in config['channel_counts'] or any(not isinstance(n,int) or not 1<=n<=128 for n in config['channel_counts']):
        raise ValueError('Include the 128-channel reference; other channel counts must be integers1–128.')
    if set(config['boundaries'])!={'PAPER_ORDER','NESTED_SAFE'}:
        raise ValueError('This paired study requires both PAPER_ORDER and NESTED_SAFE.')
    if not set(config['classifiers']).issubset(CHANNEL_MODEL_NAMES):
        raise ValueError(f'Classifiers must be chosen from {sorted(CHANNEL_MODEL_NAMES)}.')
    if not set(config['selectors']).issubset({'anova','mrmr'}):
        raise ValueError('Selectors: anova and/or mrmr.')
    if config['outer_folds']!=10 or config['inner_folds']<2:
        raise ValueError('Use ten outer folds and at least two inner folds.')
    if not isinstance(config['k_features'],int) or config['k_features']<1:
        raise ValueError('k_features must be a positive integer.')
    if config['feature_mode'] not in {'current','paper_inspired'}:
        raise ValueError('feature_mode must be current or paper_inspired.')
    if config['mrmr_relevance'] not in {'mi','f_stat'}:
        raise ValueError('mrmr_relevance must be mi or f_stat.')
    if config['channel_method'] not in {'spatial','manual'}:
        raise ValueError('channel_method must be spatial or manual.')
    if config['vmd_mode']!='off':
        raise ValueError('This controlled experiment freezes VMD off; a VMD comparison requires a separate protocol.')
    if config['run_permutation_controls']:
        for key in ('permutation_channel_counts','permutation_classifiers','permutation_selectors'):
            if not config[key] or len(config[key])!=len(set(config[key])):
                raise ValueError(f'{key} must be a nonempty list of unique values.')
        if not isinstance(config['n_permutations'],int) or config['n_permutations']<1:
            raise ValueError('n_permutations must be positive.')
        if not config['permutation_channel_counts'] or not set(config['permutation_channel_counts']).issubset(config['channel_counts']):
            raise ValueError('permutation_channel_counts must be a nonempty subset of channel_counts.')
        if not set(config['permutation_classifiers']).issubset(CHANNEL_MODEL_NAMES):
            raise ValueError('Unknown permutation classifier.')
        if not config['permutation_selectors'] or not set(config['permutation_selectors']).issubset({'anova','mrmr'}):
            raise ValueError('Unknown permutation selector.')
        if config['tune_svm']:
            raise ValueError('Freeze tune_svm=False for this initial randomized-label diagnostic.')
    if config['blas_threads']<1 or len(config['lr5_members'])!=5:
        raise ValueError('Use positive blas_threads and five LR ensemble members.')
    return config

def initialise_channel_study(config):
    config=deepcopy(config)
    validate_channel_config(config)
    config['data_root']=discover_data_root(config['data_root'])
    found=discover_subjects(config['data_root'])
    config['subjects']=found if config['subjects'] is None else config['subjects']
    if not config['subjects'] or not set(config['subjects']).issubset(found):
        raise ValueError(f'Requested subjects missing. Available subjects: {found}')
    control_subjects=config['subjects'] if config['permutation_subjects'] is None else config['permutation_subjects']
    if config['run_permutation_controls'] and (not control_subjects or not set(control_subjects).issubset(config['subjects'])):
        raise ValueError('permutation_subjects must be available in this run; None uses all selected subjects.')
    config['permutation_subjects']=control_subjects
    config['k_values']=[config['k_features']] # FixedK; inner search, when enabled, tunes SVM parameters only.
    np.random.seed(config['seed'])
    root=Path(config['output_root'])/('experiment_'+digest(dict(config=config,engine=ENGINE_ID))[:16])
    root.mkdir(parents=True,exist_ok=True)
    # Shared numerical caches survive changes to plotting/stage switches; reports remain configuration-specific.
    cache_root=Path(config['output_root'])/'evaluation_cache'/ENGINE_ID[:16]
    cache_root.mkdir(parents=True,exist_ok=True)
    save_json(root/'configuration.json',dict(config=config,engine_id=ENGINE_ID,versions=environment_info(),
              source_inspection=SOURCE_INSPECTION,channel_extension_version=CHANNEL_EXTENSION_VERSION,
              note='Processed128-channel derivatives subset in software; no re-acquisition or low-density ICA is simulated.'))
    print('Subjects:',config['subjects'],'| channels:',config['channel_counts'],'| K:',config['k_features'])
    print('Models:',config['classifiers'],'| selectors:',config['selectors'])
    print('Main:',config['run_main'],'| permutations:',config['run_permutation_controls'],config['n_permutations'])
    print('Feature mode:',config['feature_mode'],'| MRMR relevance:',config['mrmr_relevance'],'| VMD:off')
    print('Outputs:',root)
    print('PAPER_ORDER uses held-out labels in supervised selection: intentionally leaky diagnostic.')
    return config,root,cache_root

def fixed_channel_sets(datasets,config,root):
    import mne
    montage=mne.channels.make_standard_montage('biosemi128')
    canonical=list(montage.ch_names)
    positions=montage.get_positions()['ch_pos']
    if len(canonical)!=128:
        raise ValueError('Unexpected BioSemi128 montage.')
    for subject,data in datasets.items():
        if set(data['manifest']['channels'])!=set(canonical):
            raise ValueError(f'Subject{subject} channel names do not match BioSemi128. No index-based guess is used.')
        if set(data['schema'].channel)!=set(canonical):
            raise ValueError('Feature schema/channel names do not match.')
    points=np.asarray([positions[name] for name in canonical])
    # Same inspected farthest-point geometry rule, evaluated once in canonical montage order.
    start=int(np.linalg.norm(points-[0,0,points[:,2].max()],axis=1).argmin())
    chosen,remaining=[start],set(range(128))-{start}
    while remaining:
        candidates=np.asarray(sorted(remaining))
        distances=np.linalg.norm(points[candidates,None]-points[chosen][None],axis=2).min(axis=1)
        nxt=int(candidates[np.argmax(distances)])
        chosen.append(nxt)
        remaining.remove(nxt)
    counts=sorted(config['channel_counts'])
    sets={}
    for count in counts:
        if config['channel_method']=='spatial':
            membership={canonical[i] for i in chosen[:count]}
        else:
            value=config['manual_channel_sets'].get(count,config['manual_channel_sets'].get(str(count)))
            if value is None or len(value)!=count or len(set(value))!=count or not set(value).issubset(canonical):
                raise ValueError(f'manual_channel_sets[{count}] needs {count} unique valid channel names.')
            membership=set(value)
        # Canonical ordering ensures reproducible names and geometry-independent array order.
        sets[count]=[name for name in canonical if name in membership]
    for small,large in zip(counts,counts[1:]):
        if not set(sets[small]).issubset(sets[large]):
            raise ValueError('Channel configurations must be nested subsets.')
    payload=dict(method=config['channel_method'],sets={str(k):v for k,v in sets.items()},
                 canonical_order=canonical,greedy_spatial_order=[canonical[i] for i in chosen],
                 xyz_m={name:list(map(float,positions[name])) for name in canonical},
                 choice_basis='Predefined geometry/manual names only; no EEG values or labels.',
                 same_across_subjects_models_boundaries=True)
    payload['configuration_id']=digest(payload)
    save_json(Path(root)/'channel_configurations.json',payload)
    rows=[dict(channel_count=count,channel=name,canonical_index=canonical.index(name),
               x_m=float(positions[name][0]),y_m=float(positions[name][1]),z_m=float(positions[name][2]))
          for count,names in sets.items() for name in names]
    atomic_csv(pd.DataFrame(rows),Path(root)/'channel_configurations.csv')
    return sets,payload

def plot_channel_sets(channel_sets,geometry,root,show_names=False):
    positions=geometry['xyz_m']
    all_names=geometry['canonical_order']
    points=np.asarray([positions[name] for name in all_names])
    ncols=min(4,len(channel_sets))
    nrows=int(np.ceil(len(channel_sets)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(4*ncols,4*nrows),squeeze=False)
    for ax,(count,names) in zip(axes.flat,sorted(channel_sets.items(),reverse=True)):
        selected=np.array([positions[name] for name in names])
        ax.scatter(points[:,0],points[:,1],s=13,c='#dddddd')
        ax.scatter(selected[:,0],selected[:,1],s=30,c='#2878b5')
        if show_names:
            for name,(x,y,z) in zip(names,selected):
                ax.text(x,y,name,fontsize=5,ha='center',va='bottom')
        ax.set(title=f'{count} channels',xlabel='Left / Right (m)',ylabel='Posterior / Anterior (m)',aspect='equal')
        ax.grid(alpha=.15)
    for ax in list(axes.flat)[len(channel_sets):]:
        ax.axis('off')
    fig.suptitle('Fixed BioSemi 128 geometry — identical nested sets for subjects, models and boundaries')
    fig.tight_layout()
    fig.savefig(Path(root)/'channel_positions.png',dpi=160,bbox_inches='tight')
    plt.show()
    return fig

def channel_view(data,names,geometry):
    if not names or len(set(names))!=len(names):
        raise ValueError('Empty or repeated channel names.')
    mask=data['schema'].channel.isin(names).to_numpy()
    if set(data['schema'].loc[mask,'channel'])!=set(names):
        raise ValueError('Requested channel missing from feature schema.')
    indices=np.flatnonzero(mask)
    schema=data['schema'].iloc[indices].copy().reset_index(drop=True)
    schema['source_feature_index']=indices
    schema['feature_index']=np.arange(len(schema))
    manifest=deepcopy(data['manifest'])
    F=np.ascontiguousarray(data['F'][:,indices])
    manifest['parent_F_sha256']=data['manifest']['F_sha256']
    manifest['F_sha256']=array_digest(F)
    manifest['parent_feature_key']=data['feature_key']
    manifest['source_channels']=list(data['manifest']['channels'])
    manifest['channels']=[name for name in data['manifest']['channels'] if name in names]
    manifest['channel_count']=len(names)
    manifest['channel_configuration_id']=geometry['configuration_id']
    manifest['pooled_shape'][1]=len(names)
    manifest['feature_info']=dict(manifest['feature_info'],available_features=len(schema),features=len(schema))
    result=dict(data,F=F,schema=schema,manifest=manifest,
                feature_key=digest(dict(parent=data['feature_key'],channels=manifest['channels'],
                                       F_sha256=manifest['F_sha256'],geometry=geometry['configuration_id'])))
    if len(result['y'])!=len(result['F']) or not np.array_equal(result['y'],data['y']):
        raise ValueError('Channel masking changed trial/label alignment.')
    return result

def permuted_subject(data,permutation_id,config):
    if permutation_id<1:
        raise ValueError('Permutation IDs start at1; zero means the real-label experiment.')
    y=data['y'].copy()
    rng=np.random.default_rng(np.random.SeedSequence([config['seed'],int(data['subject']),int(permutation_id),104729]))
    for session,frame in data['metadata'].groupby('session',sort=True):
        rows=frame.index.to_numpy()
        y[rows]=rng.permutation(y[rows])
        assert np.array_equal(np.bincount(y[rows],minlength=4),np.bincount(data['y'][rows],minlength=4))
    metadata=data['metadata'].copy()
    metadata['original_class_id']=data['y']
    metadata['class_id']=y
    manifest=deepcopy(data['manifest'])
    manifest['permutation']=dict(permutation_id=permutation_id,within='subject/session',
                                target_sha256=array_digest(y),source_target_sha256=array_digest(data['y']))
    # Feature extraction and the channel set remain unchanged; label changes enter selection/cache identities.
    return dict(data,y=y,metadata=metadata,manifest=manifest,
                feature_key=digest(dict(parent=data['feature_key'],permutation=manifest['permutation'])))

def expected_roster(datasets,config):
    rows=[]
    if config['run_main']:
        for subject in datasets:
            for count in config['channel_counts']:
                for selector in config['selectors']:
                    for classifier in config['classifiers']:
                        for boundary in config['boundaries']:
                            rows.append(dict(subject=subject,channel_count=count,selector=selector,classifier=classifier,
                                             boundary=boundary,permutation_id=0,experiment='main'))
    if config['run_permutation_controls']:
        for subject in config['permutation_subjects']:
            for permutation_id in range(1,config['n_permutations']+1):
                for count in config['permutation_channel_counts']:
                    for selector in config['permutation_selectors']:
                        for classifier in config['permutation_classifiers']:
                            for boundary in config['boundaries']:
                                rows.append(dict(subject=subject,channel_count=count,selector=selector,classifier=classifier,
                                                 boundary=boundary,permutation_id=permutation_id,experiment='permutation'))
    return pd.DataFrame(rows,columns=['subject','channel_count','selector','classifier','boundary','permutation_id','experiment'])

def preflight_channel_inputs(datasets,channel_sets,geometry,config,root):
    rows=[]
    plans={}
    for subject,data in datasets.items():
        if not data['metadata'].index.equals(pd.RangeIndex(len(data['metadata']))):
            raise ValueError('Metadata must use the same positional row order as F/y.')
        plans[subject]=cv_plan(data,config)
        save_json(Path(root)/f'sub-{subject:02d}_real_label_plan.json',plans[subject])
        for count,names in channel_sets.items():
            view=channel_view(data,names,geometry)
            if config['k_features']>view['F'].shape[1]:
                raise ValueError(f'K={config["k_features"]} exceeds {view["F"].shape[1]} candidates at{count} channels. Lower K or change representation before training.')
            assert cv_plan(view,config)==plans[subject]
            rows.append(dict(subject=subject,channel_count=count,trials=len(view['y']),
                             available_features=view['F'].shape[1],selected_features=config['k_features'],
                             features_per_channel=view['F'].shape[1]//count,shape=str(view['F'].shape)))
    preview=pd.DataFrame(rows)
    atomic_csv(preview,Path(root)/'feature_dimensions.csv')
    roster=expected_roster(datasets,config)
    atomic_csv(roster,Path(root)/'requested_roster.csv')
    print(f'Planned completed-subject evaluations:{len(roster)}; outer classifier fits:{len(roster)*config["outer_folds"]} '
          '(inner tuning, when enabled, adds fits).')
    return preview,roster,plans

def evaluate_channel_task(view,selector,classifier,boundary,permutation_id,config,cache_root,active):
    count=view['manifest']['channel_count']
    spec=dict(stage='channel_reduction',selector=selector,k=config['k_features'],classifier=classifier,
              channel_count=count,permutation_id=permutation_id,
              channel_configuration_id=view['manifest']['channel_configuration_id'],
              candidate_features=view['F'].shape[1],label_condition='real' if permutation_id==0 else 'permuted')
    result=evaluate(view,spec,boundary,config,cache_root)
    original_indices=view['schema'].source_feature_index.to_numpy()
    result['selections']=result['selections'].copy()
    result['selections']['source_feature_index']=original_indices[result['selections'].feature_index.to_numpy()]
    used=result['selections'].groupby('fold').channel.nunique()
    if not set(result['selections'].channel).issubset(view['manifest']['channels']) or used.max()>count:
        raise ValueError('Selected features use channels outside the fixed configuration.')
    result['row']['selected_channels_min']=int(used.min())
    result['row']['selected_channels_max']=int(used.max())
    result['row']['selected_channels_mean']=float(used.mean())
    result['row']['mean_train_accuracy']=float(result['fold_scores'].train_accuracy.mean())
    result['row']['elapsed_seconds_sum']=float(sum(a['elapsed_seconds'] for a in result['audits']))
    # Enrich resumed/recomputed outputs with the source feature mapping and fixed channel names.
    atomic_csv(result['selections'],result['directory']/'selected_features.csv')
    save_json(result['directory']/'channel_input.json',dict(subject=view['subject'],channel_count=count,
              channels=view['manifest']['channels'],configuration_id=spec['channel_configuration_id'],
              feature_key=view['feature_key'],F_sha256=view['manifest']['F_sha256'],
              parent_feature_key=view['manifest']['parent_feature_key'],permutation=view['manifest'].get('permutation'),
              trial_ids=view['metadata'].trial_id.tolist(),scored_target_sha256=array_digest(view['y'])))
    save_json(result['directory']/'result.json',result['row'])
    active[result['row']['result_id']]=result
    return result

def run_channel_stage(datasets,channel_sets,geometry,config,root,cache_root,active,status,experiment='main'):
    if experiment not in {'main','permutation'}:
        raise ValueError('experiment must be main or permutation.')
    if experiment=='main':
        subjects=list(datasets); ids=[0]; counts=config['channel_counts']
        selectors=config['selectors']; classifiers=config['classifiers']
    else:
        subjects=config['permutation_subjects']; ids=list(range(1,config['n_permutations']+1))
        counts=config['permutation_channel_counts']; selectors=config['permutation_selectors']; classifiers=config['permutation_classifiers']
    for subject in subjects:
        for permutation_id in ids:
            original=datasets[subject]
            data=original if permutation_id==0 else permuted_subject(original,permutation_id,config)
            plan=cv_plan(data,config)
            if permutation_id:
                save_json(Path(root)/f'sub-{subject:02d}_permutation-{permutation_id:03d}_plan.json',plan)
                target=data['metadata'][['trial_id','subject','session','original_class_id','class_id']].copy()
                target['permutation_id']=permutation_id
                atomic_csv(target,Path(root)/f'sub-{subject:02d}_permutation-{permutation_id:03d}_labels.csv')
            for count in counts:
                view=channel_view(data,channel_sets[count],geometry)
                assert cv_plan(view,config)==plan
                print(f'\n{experiment.upper()} | subject{subject:02d} | channels={count} | candidates={view["F"].shape[1]} | '
                      f'K={config["k_features"]} | permutation={permutation_id}',flush=True)
                for selector in selectors:
                    for classifier in classifiers:
                        if classifier=='lgbm' and importlib.util.find_spec('lightgbm') is None:
                            for boundary in config['boundaries']:
                                status.append(dict(subject=subject,channel_count=count,selector=selector,classifier=classifier,
                                                   boundary=boundary,permutation_id=permutation_id,experiment=experiment,
                                                   status='skipped',reason='LightGBM unavailable; no substitute result.'))
                            print('LightGBM unavailable: explicitly skipped.',flush=True)
                            continue
                        paired=[]
                        for boundary in config['boundaries']:
                            result=evaluate_channel_task(view,selector,classifier,boundary,permutation_id,config,cache_root,active)
                            paired.append(result['oof'])
                            atomic_csv(pd.DataFrame([r['row'] for r in active.values()]),Path(root)/'active_subject_results.csv')
                        assert paired[0].trial_id.tolist()==paired[1].trial_id.tolist()
                        assert np.array_equal(paired[0].y_true,paired[1].y_true)
                        assert np.array_equal(paired[0].outer_fold,paired[1].outer_fold)
    atomic_csv(pd.DataFrame(status,columns=['subject','channel_count','selector','classifier','boundary','permutation_id','experiment','status','reason']),
               Path(root)/'stage_status.csv')
    return pd.DataFrame([r['row'] for r in active.values() if (r['row']['permutation_id']==0)==(experiment=='main')])

def channel_aggregates(scores):
    columns=['feature_mode','selector','k','classifier','channel_count','boundary','selection_boundary','n_subjects',
             'accuracy_mean','accuracy_sd','balanced_accuracy_mean','balanced_accuracy_sd','macro_f1_mean','macro_f1_sd']
    if scores.empty:
        return pd.DataFrame(columns=columns)
    keys=columns[:7]
    rows=[]
    for setting,frame in scores.groupby(keys,dropna=False):
        if frame.subject.duplicated().any():
            raise ValueError('Subject aggregation received repeated subjects; do not treat permutations/folds as subjects.')
        row=dict(zip(keys,setting),n_subjects=frame.subject.nunique(),subjects=','.join(map(str,sorted(frame.subject))))
        for metric in ('accuracy','balanced_accuracy','macro_f1',*[f'recall_{i}' for i in LABELS]):
            row[metric+'_mean']=frame[metric].mean()
            row[metric+'_sd']=frame[metric].std(ddof=1)
        row['mean_train_accuracy']=frame.mean_train_accuracy.mean()
        row['mean_fit_seconds_per_subject']=frame.elapsed_seconds_sum.mean()
        rows.append(row)
    return pd.DataFrame(rows)

def paired_boundary_table(scores):
    keys=['subject','feature_mode','selector','k','classifier','channel_count','permutation_id']
    if scores.empty:
        return pd.DataFrame(columns=keys)
    a=scores[scores.boundary=='PAPER_ORDER']; b=scores[scores.boundary=='NESTED_SAFE']
    merged=a.merge(b,on=keys,suffixes=('_paper_order','_nested_safe'),validate='one_to_one')
    columns=keys.copy()
    for metric in ('accuracy','balanced_accuracy','macro_f1'):
        merged[metric+'_gap']=merged[metric+'_paper_order']-merged[metric+'_nested_safe']
        columns += [metric+'_paper_order',metric+'_nested_safe',metric+'_gap']
    merged['paper_order_label']=PAPER_LABEL
    return merged[columns+['paper_order_label']]

def changes_from_128(scores):
    keys=['subject','feature_mode','selector','k','classifier','boundary','permutation_id']
    if scores.empty:
        return pd.DataFrame(columns=keys)
    reference=scores[scores.channel_count==128]
    columns=keys+['accuracy','balanced_accuracy','macro_f1']
    merged=scores.merge(reference[columns],on=keys,suffixes=('','_128'),validate='many_to_one')
    for metric in ('accuracy','balanced_accuracy','macro_f1'):
        merged[metric+'_change_from_128']=merged[metric]-merged[metric+'_128']
    return merged[keys+['channel_count','selection_boundary']+
                  [name for metric in ('accuracy','balanced_accuracy','macro_f1') for name in (metric,metric+'_128',metric+'_change_from_128')]]

def export_channel_tables(active,status,roster,config,root):
    score_columns=['result_id','subject','feature_mode','scope','stage','selector','k','classifier','channel_count',
                   'permutation_id','boundary','selection_boundary','accuracy','balanced_accuracy','macro_f1',
                   'mean_train_accuracy','elapsed_seconds_sum']
    scores=pd.DataFrame([r['row'] for r in active.values()]) if active else pd.DataFrame(columns=score_columns)
    main=scores[scores.permutation_id==0].copy()
    controls=scores[scores.permutation_id>0].copy()
    summary=channel_aggregates(main)
    gaps=paired_boundary_table(main)
    changes=changes_from_128(main)
    check_keys=['subject','channel_count','selector','classifier','boundary','permutation_id']
    completed=scores[check_keys].copy()
    completed['status']='complete'
    roster_status=roster.merge(completed,on=check_keys,how='left',validate='one_to_one')
    if status:
        skipped=pd.DataFrame(status).drop_duplicates(check_keys,keep='last')
        roster_status=roster_status.merge(skipped[check_keys+['reason']],on=check_keys,how='left')
        roster_status.loc[roster_status.reason.notna(),'status']='skipped'
    roster_status['status']=roster_status.status.fillna('pending')
    if not summary.empty:
        summary['requested_subjects']=len(config['subjects'])
        summary['complete_subject_roster']=summary.n_subjects.eq(len(config['subjects']))
    by_permutation=[]
    for pid,frame in controls.groupby('permutation_id'):
        table=channel_aggregates(frame)
        table['permutation_id']=pid
        by_permutation.append(table)
    control_means=pd.concat(by_permutation,ignore_index=True) if by_permutation else pd.DataFrame()
    control_summary=[]
    control_keys=['feature_mode','selector','k','classifier','channel_count','boundary','selection_boundary']
    if not control_means.empty:
        for setting,frame in control_means.groupby(control_keys):
            control_summary.append(dict(zip(control_keys,setting),n_permutations=frame.permutation_id.nunique(),
                n_subjects=int(frame.n_subjects.min()),mean_balanced_accuracy=frame.balanced_accuracy_mean.mean(),
                sd_across_permutation_means=frame.balanced_accuracy_mean.std(ddof=1),
                min_balanced_accuracy=frame.balanced_accuracy_mean.min(),max_balanced_accuracy=frame.balanced_accuracy_mean.max()))
    control_summary=pd.DataFrame(control_summary)
    session_rows=[]
    for result in active.values():
        row=result['row']
        for session,group in result['oof'].groupby('session'):
            session_rows.append(dict(**{k:row[k] for k in ('result_id','subject','feature_mode','selector','k','classifier',
                  'channel_count','permutation_id','boundary','selection_boundary')},session=int(session),n_trials=len(group),
                  **metrics(group.y_true,group.y_pred)))
    sessions=pd.DataFrame(session_rows)
    for name,frame in [('per_subject_scores',main),('subject_mean_scores',summary),('paired_boundary_gaps',gaps),
                       ('changes_from_128',changes),('permutation_subject_scores',controls),
                       ('permutation_cohort_means',control_means),('permutation_summary',control_summary),
                       ('session_oof_scores',sessions),('completion_status',roster_status)]:
        atomic_csv(frame,Path(root)/(name+'.csv'))
    lines=['Scores are proportions. Means weight subjects equally. SD describes subjects, not independent CV folds.',
           'PAPER_ORDER uses held-out labels in selection and is an intentionally leaky diagnostic.',
           'NESTED_SAFE estimates held-out performance under the declared subject-specific pooled-session protocol.',
           'This is channel masking of processed high-density recordings, not a simulation of new low-density recordings.']
    for setting,frame in summary.groupby(['feature_mode','selector','classifier']) if not summary.empty else []:
        safe=frame[frame.boundary=='NESTED_SAFE'].sort_values('channel_count')
        lines.append(f'{setting}: training-only channel-count/BA means = '+str(dict(zip(safe.channel_count,safe.balanced_accuracy_mean.round(4)))))
        base=safe[safe.channel_count==128]
        if len(base) and abs(base.balanced_accuracy_mean.iloc[0]-.25)<=.05:
            lines.append(f'{setting}: 128-channel mean is near the25% reference; a flat reduction curve does not establish preservation of useful decoding.')
    if not control_summary.empty:
        lines.append('Randomized-label results are an initial diagnostic. A few permutations are not a formal significance test; permutation repetitions are not independent subjects.')
        for row in control_summary.itertuples(index=False):
            lines.append(f'Randomized labels, {row.classifier}/{row.selector}, {row.channel_count} channels, {row.selection_boundary}: '
                         f'mean BA={row.mean_balanced_accuracy:.1%} across{row.n_permutations} permutations and{row.n_subjects} subjects.')
    if roster_status.status.eq('pending').any():
        lines.append(f'{int(roster_status.status.eq("pending").sum())} requested evaluations remain pending. Partial results are labeled in completion_status.csv.')
    (Path(root)/'interpretation.txt').write_text('\n\n'.join(lines),encoding='utf-8')
    save_json(Path(root)/'run_manifest.json',dict(config=config,engine_id=ENGINE_ID,versions=environment_info(),
               source_inspection=SOURCE_INSPECTION,active_result_ids=list(active),expected_evaluations=len(roster),
               completed_evaluations=len(scores),skipped_evaluations=len(status),
               label_control_note='Labels randomized within subject/session, then selection, fitting and scoring use those labels. '
                                  'Each permutation is restratified once; its plan is shared across budgets and modes.'))
    for line in lines:
        print(line+'\n')
    return dict(subject_scores=main,summary=summary,gaps=gaps,changes=changes,
                controls=controls,control_summary=control_summary,sessions=sessions,completion=roster_status)

def plot_channel_performance(tables,root):
    means=tables['summary']
    if means.empty:
        print('No real-label results yet.')
        return
    for setting,frame in means.groupby(['feature_mode','selector','classifier']):
        fig,axes=plt.subplots(1,2,figsize=(13,4.8))
        colors={'PAPER_ORDER':'#d95f02','NESTED_SAFE':'#1b9e77'}
        for boundary,group in frame.groupby('boundary'):
            group=group.sort_values('channel_count')
            axes[0].errorbar(group.channel_count,100*group.balanced_accuracy_mean,
                            yerr=100*group.balanced_accuracy_sd.fillna(0),marker='o',capsize=3,
                            color=colors[boundary],label=boundary_label(boundary))
        mode,selector,classifier=setting
        gaps=tables['gaps']
        g=gaps[(gaps.feature_mode==mode)&(gaps.selector==selector)&(gaps.classifier==classifier)]
        gapmeans=g.groupby('channel_count').balanced_accuracy_gap.agg(['mean','std']).sort_index()
        axes[1].errorbar(gapmeans.index,100*gapmeans['mean'],yerr=100*gapmeans['std'].fillna(0),marker='o',capsize=3,color='#7570b3')
        axes[0].axhline(25,linestyle=':',color='#444444',label='Four-class chance reference')
        axes[0].set(ylabel='Balanced accuracy (%)',title='Subject mean ± SD (when at least two subjects)',ylim=(0,103))
        axes[1].set(ylabel='PAPER_ORDER − NESTED_SAFE (percentage points)',title='Paired subject boundary gap ± subject SD')
        axes[1].axhline(0,linestyle=':',color='#444444')
        for ax in axes:
            ax.set(xlabel='Available EEG channels',xticks=sorted(frame.channel_count.unique()))
            ax.grid(alpha=.2)
        axes[0].legend(fontsize=7,loc='best')
        fig.suptitle(f'{mode}, {selector.upper()}, {classifier.upper()} — fixed selected K, matched trials/folds; '
                     f'completed subjects per point: {int(frame.n_subjects.min())}–{int(frame.n_subjects.max())}')
        fig.tight_layout()
        fig.savefig(Path(root)/f'performance_{mode}_{selector}_{classifier}.png',dpi=150,bbox_inches='tight')
        plt.show()

def plot_changes_from_128(tables,root):
    changes=tables['changes']
    if changes.empty:
        return
    for setting,frame in changes.groupby(['feature_mode','selector','classifier']):
        fig,ax=plt.subplots(figsize=(8,4.8))
        for boundary,group in frame.groupby('boundary'):
            mean=group.groupby('channel_count').balanced_accuracy_change_from_128.mean().sort_index()
            ax.plot(mean.index,100*mean,marker='o',color=CHANNEL_BOUNDARY_COLORS[boundary],label=boundary_label(boundary))
        ax.axhline(0,color='#444444',linestyle=':')
        ax.set(xlabel='Available EEG channels',ylabel='BA change from 128 (percentage points)',
               xticks=sorted(frame.channel_count.unique()),title=f'{" | ".join(setting)}: paired changes from each subject’s 128-channel score')
        ax.grid(alpha=.2); ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(Path(root)/('change_from128_'+'_'.join(setting)+'.png'),dpi=150,bbox_inches='tight')
        plt.show()

def plot_channel_subjects(tables,root):
    scores=tables['subject_scores']
    if scores.empty:
        return
    for setting,frame in scores.groupby(['feature_mode','selector','classifier']):
        subjects=sorted(frame.subject.unique())
        ncols=min(3,len(subjects)); nrows=int(np.ceil(len(subjects)/ncols))
        fig,axes=plt.subplots(nrows,ncols,figsize=(5*ncols,3.6*nrows),squeeze=False)
        for ax,subject in zip(axes.flat,subjects):
            for boundary,group in frame[frame.subject==subject].groupby('boundary'):
                group=group.sort_values('channel_count')
                ax.plot(group.channel_count,100*group.balanced_accuracy,marker='o',color=CHANNEL_BOUNDARY_COLORS[boundary],label=boundary_label(boundary))
            ax.axhline(25,color='#444444',linestyle=':')
            ax.set(title=f'Subject {subject:02d}',xlabel='Channels',ylabel='BA (%)',ylim=(0,103),xticks=sorted(frame.channel_count.unique()))
            ax.grid(alpha=.2)
        for ax in list(axes.flat)[len(subjects):]:
            ax.axis('off')
        axes.flat[0].legend(fontsize=6)
        fig.suptitle(f'{" | ".join(setting)}: subject-specific channel-reduction curves')
        fig.tight_layout()
        fig.savefig(Path(root)/('subjects_'+'_'.join(setting)+'.png'),dpi=140,bbox_inches='tight')
        plt.show()

def plot_permutation_controls(tables,root):
    controls=tables['control_summary']
    if controls.empty:
        print('Permutation controls disabled or not completed.')
        return
    for setting,frame in controls.groupby(['feature_mode','selector','classifier']):
        fig,ax=plt.subplots(figsize=(9,4.8))
        for boundary,group in frame.groupby('boundary'):
            group=group.sort_values('channel_count')
            ax.plot(group.channel_count,100*group.mean_balanced_accuracy,marker='o',color=CHANNEL_BOUNDARY_COLORS[boundary],label=boundary_label(boundary))
            ax.fill_between(group.channel_count,100*group.min_balanced_accuracy,100*group.max_balanced_accuracy,color=CHANNEL_BOUNDARY_COLORS[boundary],alpha=.12)
        ax.axhline(25,linestyle=':',color='#444444')
        ax.set(xlabel='EEG channels',ylabel='BA against randomized labels (%)',ylim=(0,103),
               xticks=sorted(frame.channel_count.unique()),title=f'{" | ".join(setting)}: shuffled labels\nShading: min–max permutation means; not a confidence interval')
        ax.grid(alpha=.2); ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(Path(root)/('randomized_labels_'+'_'.join(setting)+'.png'),dpi=150,bbox_inches='tight')
        plt.show()

def export_channel_review(active,config,root):
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    target=Path(config['output_root'])/f'channel_reduction_review_{stamp}.zip'
    root=Path(root)
    # Reports belong to the current config. Include only currently active cached runs.
    with ZipFile(target,'w',ZIP_DEFLATED) as archive:
        for path in sorted(root.iterdir()):
            if path.is_file() and path.suffix in {'.csv','.json','.txt','.png','.py'}:
                archive.write(path,path.name)
        for result in active.values():
            for path in sorted(result['directory'].iterdir()):
                if path.is_file() and path.suffix in {'.csv','.json'}:
                    archive.write(path,f'runs/{result["row"]["result_id"]}/{path.name}')
    print(f'Review ZIP:{target} ({target.stat().st_size/1024**2:.2f} MB). No raw EEG or feature arrays included.')
    return target


## 7. Validate settings and initialize the active run

Rerun this cell after editing CONFIG. Evaluation caches remain reusable for exact matching settings; report directories are separated by full configuration. The active registry contains only evaluations run/resumed in this execution. No old results are silently counted as completed.


In [ ]:
RUN_CONFIG, RUN_ROOT, CACHE_ROOT = initialise_channel_study(CONFIG)
ACTIVE_RESULTS = {}
STAGE_STATUS = []
print("Versions:", environment_info())
save_json(RUN_ROOT / "source_inspection.json", SOURCE_INSPECTION)


## 8. Load INNER SPEECH data and inspect the recording summary

`DATASETS[subject]` provides `F`,`y`,`metadata`,`schema`,`manifest` and `feature_key`. All sessions are pooled within a subject; different subjects never share a trained model. These are subject-dependent random-fold evaluations, not LOSO or leave-one-session-out.


In [ ]:
DATASETS = {}
for subject in RUN_CONFIG["subjects"]:
    DATASETS[subject] = prepare_subject_replication(subject, RUN_CONFIG)
DATASET_SUMMARY = dataset_summary(DATASETS, RUN_ROOT)
display(DATASET_SUMMARY)
subject_data = DATASETS[RUN_CONFIG["subjects"][0]]
F = subject_data["F"]
y = subject_data["y"]
trial_metadata = subject_data["metadata"]
print("First subject:", subject_data["subject"], "| F:", F.shape, "| class counts:", np.unique(y, return_counts=True))


## 9. Freeze and visualize the channel sets

Inspect electrode placement and the exported channel-name lists. The same names apply to every subject/model/boundary. Do not revise sets based on their test accuracy. The plot is a standard-montage XY view; it is not a feature-importance map.


In [ ]:
CHANNEL_SETS, CHANNEL_GEOMETRY = fixed_channel_sets(DATASETS, RUN_CONFIG, RUN_ROOT)
display(pd.DataFrame([{ "channel_count": n, "channels": ", ".join(names)}
                      for n, names in CHANNEL_SETS.items()]))
if RUN_CONFIG["make_plots"]:
    plot_channel_sets(CHANNEL_SETS, CHANNEL_GEOMETRY, RUN_ROOT, RUN_CONFIG["show_channel_names"])


## 10. Check feature dimensions, trial alignment and expected work

The preview verifies that channel reduction does not change trial IDs/targets/folds. It checks that the requested K fits each budget. Training-time constant eligibility is checked again by each fitted selector. The requested roster shows how many subject evaluations/fits are planned, and later records complete/skipped/pending status.


In [ ]:
FEATURE_DIMENSIONS, REQUESTED_ROSTER, BASE_PLANS = preflight_channel_inputs(
    DATASETS, CHANNEL_SETS, CHANNEL_GEOMETRY, RUN_CONFIG, RUN_ROOT)
plan = BASE_PLANS[subject_data["subject"]]
display(FEATURE_DIMENSIONS)
display(REQUESTED_ROSTER.groupby(["experiment", "classifier", "selector", "channel_count", "boundary"])
        .size().rename("subject_evaluations").reset_index())


## 11. Real-label channel-reduction comparison

For each subject/channel budget/selector/classifier, both boundaries share the same 10-fold plan. PAPER_ORDER ranks features using all subject labels before CV; NESTED_SAFE ranks on outer train only. Models use fixed K=590 by default. Supports, OOF predictions, train metrics, confusion matrices and audits are saved per evaluation.

The128-channel result should agree with the previous experiment when data, versions, folds and numerical settings agree. Channel sets do not inherit the 128-channel selected feature list: selection is rerun for each restricted pool. Resume checks exact data/target/configuration identities and completes remaining folds.


In [ ]:
if RUN_CONFIG["run_main"]:
    MAIN_RESULTS = run_channel_stage(DATASETS, CHANNEL_SETS, CHANNEL_GEOMETRY, RUN_CONFIG,
                                    RUN_ROOT, CACHE_ROOT, ACTIVE_RESULTS, STAGE_STATUS, experiment="main")
    if not MAIN_RESULTS.empty:
        display(MAIN_RESULTS[["subject", "channel_count", "selector", "classifier", "k", "selection_boundary",
                              "accuracy", "balanced_accuracy", "macro_f1", "selected_channels_mean"]])
    else:
        print("No completed main results; check stage_status.csv for unavailable optional classifiers.")
else:
    print("Main experiment disabled; set run_main=True.")
# Persist the current report even before controls finish. Pending controls stay explicitly marked.
CHANNEL_TABLES = export_channel_tables(ACTIVE_RESULTS, STAGE_STATUS, REQUESTED_ROSTER, RUN_CONFIG, RUN_ROOT)
display(CHANNEL_TABLES["summary"])


## 12. Randomized-label diagnostic at 128 and 16 channels

Labels are shuffled within each subject/session before any supervised ranking/model fit. Original labels remain in a separate metadata column. Selection, fitting and scoring all use the randomized targets; feature matrices and spatial channel configurations stay unchanged.

For each permutation, a reproducible stratified 10-fold plan is created from its randomized targets and reused across channel budgets and both boundaries. That control plan can differ from the real-label plan. Class counts are preserved within sessions. A few permutations illustrate a failure mechanism; they are not a formal significance test or independent participants.

If before-CV selection yields high scores on randomized targets while the training-only route remains near chance, it demonstrates that this diagnostic procedure can produce apparent decoding without the original label association. No outcome is assumed in advance.


In [ ]:
if RUN_CONFIG["run_permutation_controls"]:
    PERMUTATION_RESULTS = run_channel_stage(DATASETS, CHANNEL_SETS, CHANNEL_GEOMETRY, RUN_CONFIG,
        RUN_ROOT, CACHE_ROOT, ACTIVE_RESULTS, STAGE_STATUS, experiment="permutation")
    if not PERMUTATION_RESULTS.empty:
        display(PERMUTATION_RESULTS[["subject", "permutation_id", "channel_count", "selector", "classifier",
                                    "selection_boundary", "balanced_accuracy", "macro_f1"]])
    else:
        print("No completed control results; check stage_status.csv for unavailable optional classifiers.")
else:
    print("Permutation controls disabled. Enable run_permutation_controls to test the label-association control.")


## 13. Summary tables and complete/partial status

CSV scores are proportions: 0.30 means 30%. Means give subjects equal weight; SD measures between-subject variation, not independent folds. The boundary-gap table pairs the same subject/budget/model setting; the 128-channel change table compares each subject against its own128-channel score within the same boundary. Randomized-label means are first computed across subjects for each permutation, then summarized across permutations; repetitions are never treated as extra subjects.

Session OOF scores are retained with feature mode/channel count/boundary/permutation labels. Pending and skipped models are visible. A flat curve near 25% does not establish preservation of useful decoding; it may simply preserve chance-level performance.


In [ ]:
CHANNEL_TABLES = export_channel_tables(ACTIVE_RESULTS, STAGE_STATUS, REQUESTED_ROSTER, RUN_CONFIG, RUN_ROOT)
print("Subject means")
display(CHANNEL_TABLES["summary"])
print("Paired boundary gaps")
display(CHANNEL_TABLES["gaps"])
print("Changes from each subject's128-channel reference")
display(CHANNEL_TABLES["changes"])
print("Randomized-label controls")
display(CHANNEL_TABLES["control_summary"])
print("Completion status")
display(CHANNEL_TABLES["completion"].groupby(["experiment", "status"]).size().rename("evaluations").reset_index())


## 14. Channel-count curves, paired drops and randomized-label plots

The main curve plots subject mean BA ± subject sample SD for each boundary. The adjacent panel plots the paired before-CV-minus-training-only gap. Additional plots show changes from 128 and individual subjects. The randomized-label plot shades min–max permutation means, not a confidence interval. Avoid selecting a final channel set/K/model from these outer-test plots; predeclare a follow-up confirmation protocol.


In [ ]:
if RUN_CONFIG["make_plots"]:
    plot_channel_performance(CHANNEL_TABLES, RUN_ROOT)
    plot_changes_from_128(CHANNEL_TABLES, RUN_ROOT)
    if RUN_CONFIG["plot_individual_subjects"]:
        plot_channel_subjects(CHANNEL_TABLES, RUN_ROOT)
    plot_permutation_controls(CHANNEL_TABLES, RUN_ROOT)


## 15. Download the review ZIP

The bundle includes the active OOF predictions, selected feature/channel/source-index records, confusion matrices, fit/test/inner/bootstrap audit JSON, fixed electrode configurations, real/randomized label plans, control labels, configuration, completion status, subject/session tables and plots. It excludes raw EEG, large feature matrices and binary checkpoints.

Download this ZIP from Kaggle output and send it for analysis. Rerun the tables/export cells after any resumed stage so the bundle reflects currently completed evaluations.


In [ ]:
REVIEW_ZIP = export_channel_review(ACTIVE_RESULTS, RUN_CONFIG, RUN_ROOT)
display(FileLink(str(REVIEW_ZIP)))


## How to configure the next run

1. Keep `subjects=[1]` initially; verify dimensions, electrode sets and paired folds.
2. Use `subjects=None` for the full cohort. To keep the first label-control run small, set `permutation_subjects=[1]`; otherwise controls run on the whole selected cohort.
3. Add classifiers in `classifiers`, for example `["svm","lda","lr","lr5"]`. Keep `models` as the hyperparameter dictionary.
4. Add `"anova"` to `selectors` for a matched selector experiment; define permutation selectors/models separately to avoid an unnecessarily large control run.
5. Disable controls temporarily with `run_permutation_controls=False`; their code remains available. Exact caches are reusable when they are enabled again.
6. Change K only as a separate declared comparison. Feature count is not channel count. Bandpower-only features require a smaller K.

Rerun setup and downstream cells after configuration changes. The helpers preserve the familiar `DATASETS`,`subject_data`,`F`,`y`,`trial_metadata`,`plan` variables, plus `CHANNEL_SETS`,`CHANNEL_TABLES`,`ACTIVE_RESULTS` and `REVIEW_ZIP`. Neither boundary automatically chooses the best electrode set from outer scores.
